# GSPO: STEM Reasoning via Verifiable Rewards + Curriculum Learning

This notebook applies **GSPO** (Group Sequence Policy Optimization) with verifiable
rewards and **curriculum learning** to improve STEM reasoning quality.

**Training pipeline stage:** 1 of 3 (**GSPO (curriculum)** → KTO → DPO)

**Target hardware:** Google Colab G4 RTX PRO 6000 (~102GB) / A100 80GB / A100 40GB (auto-detect)

**Key strategy:** `enable_thinking=True` + `ThinkingBudgetProcessor(1536)` + `Cold-Start SFT(200 steps)`
- Model NEEDS chain-of-thought (`<think>`) to solve STEM problems (verified: accuracy ~0% without thinking)
- TRL generation has no built-in thinking budget → `ThinkingBudgetProcessor` forces `</think>` after 1536 tokens
- Cold-Start SFT (DeepSeek-R1 approach): 200 SFT steps on rejection-sampled correct solutions before RL
- ReDit dithering (sigma=0.05) creates continuous gradient landscape — NO zero-variance masking (mutually exclusive)

**Key features:**
- `importance_sampling_level="sequence"` — GSPO sequence-level importance ratios (Qwen3 training)
- `loss_type="dr_grpo"` — Dr. GRPO constant length normalization (arXiv 2503.20783)
- `beta=0.0` — No KL regularization (GSPO/DAPO standard)
- `epsilon=3e-4`, `epsilon_high=4e-4` — GSPO sequence-level clipping (arXiv 2507.18071, Section 5.1)
- `mask_truncated_completions=False` — truncated completions give 0 correctness (negative signal)
- `enable_thinking=True` + ThinkingBudgetProcessor — bounded CoT for STEM reasoning
- `steps_per_generation=8`, `gradient_accumulation_steps=8`
- G=8 completions per prompt, **LoRA r=16** (optimal for RL per Tina paper, arXiv 2504.15777)
- **ReDit reward dithering (arXiv 2506.18631):** sigma=0.05 Gaussian noise for continuous gradients
- **GDPO-style decoupled reward normalization:** correctness, format normalized independently (socratic removed — near-zero signal)
- **Curriculum learning (GRPO-LEAD):** Stage 1 trains easy+medium, Stage 2 all with difficulty reweighting
- **Cold-Start SFT (DeepSeek-R1):** 200 SFT steps before RL to raise baseline from ~5% to 15-30%

**Research basis:** See `research/findings_grpo_thinking_training_2026-03-19.md`

In [1]:
# ============================================================
# ALL dependencies in one shot. After this cell: RESTART RUNTIME.
# After restart: SKIP this cell, start from Cell 2.
# ============================================================
import os, subprocess, sys

# Step 1: Unsloth (core training framework)
!pip install -q --upgrade --force-reinstall --no-cache-dir unsloth unsloth_zoo

# Step 2: Transformers v5 (required for Qwen3.5 hybrid architecture) + ML
!pip install -q "transformers>=5.0.0" trl peft datasets

# Step 3: torchvision MUST match torch version (torch 2.11 -> torchvision 0.23+)
# Old torchvision (0.22) + Pillow 12 = ImportError: '_Ink' from 'PIL._typing'
# See: github.com/unslothai/unsloth/issues/3475
!pip install -q --upgrade scipy "torchvision>=0.23.0"

# Step 4: Other dependencies
!pip install -q accelerate bitsandbytes sentencepiece protobuf loguru
!pip install -q sympy chempy

# Step 5: DeltaNet acceleration (Triton kernels via flash-linear-attention)
# Qwen3.5-9B uses GatedDeltaNet in 24/32 layers. FLA provides JIT-compiled
# Triton kernels for these layers. causal-conv1d is NOT needed (FLA has its own).
!pip install -q flash-linear-attention 2>&1 | tail -3

from huggingface_hub import login
login()

print()
print("=" * 60)
print("  RESTART RUNTIME NOW: Runtime -> Restart session")
print("  After restart: SKIP this cell, run Cell 2 onwards.")
print("=" * 60)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 41.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 31.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 537.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 107.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 542.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.6/79.6 kB 85.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 MB 231.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.6/401.6 kB 884.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 879.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 239.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 750.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(



  RESTART RUNTIME NOW: Runtime -> Restart session
  After restart: SKIP this cell, run Cell 2 onwards.


In [2]:
# NOTE: llm_blender, weave, mergekit — deferred to merge cells (27-28).
# Installing here breaks pydantic/safetensors/protobuf (Cell 11 warning).
# Run ONLY before Cell 27: !pip install llm_blender weave mergekit
pass


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.1/92.1 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 955.0/955.0 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.9/45.9 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.2/102.2 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.9/104.9 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.7/354.7 kB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 76.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 431.7/431.7 kB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
# ============================================================
# Environment + sys.path (runs AFTER restart — Cell 1 is skipped)
# ============================================================
import sys, os

# CRITICAL: Must be set BEFORE importing unsloth
os.environ["UNSLOTH_OFFLOAD_GRADIENTS"] = "0"  # Keep gradients on GPU (we have headroom)

# Fix bitsandbytes CUDA library path (libnvJitLink.so.13)
# Must be set BEFORE any bitsandbytes native call
import glob as _glob
_nvjit_paths = _glob.glob("/usr/local/lib/python3.*/dist-packages/nvidia/nvjitlink/lib")
_nvjit_paths += _glob.glob("/usr/local/lib/python3.*/dist-packages/nvidia/cuda_runtime/lib")
for _p in _nvjit_paths:
    if _p not in os.environ.get("LD_LIBRARY_PATH", ""):
        os.environ["LD_LIBRARY_PATH"] = os.environ.get("LD_LIBRARY_PATH", "") + ":" + _p
        print(f"  Added to LD_LIBRARY_PATH: {_p}")
# Also run ldconfig so the dynamic linker sees them
os.system("ldconfig /usr/local/lib/python3.*/dist-packages/nvidia/*/lib/ 2>/dev/null")
# Do NOT set CUDA_LAUNCH_BLOCKING=1 — it serializes GPU ops and kills performance

DRIVE_ROOT = "/content/drive/MyDrive"
if DRIVE_ROOT not in sys.path:
    sys.path.insert(0, DRIVE_ROOT)

for s in ["training/scripts/stem_rewards.py", "training/scripts/verify_answers.py", "training/scripts/sort_curriculum.py"]:
    print(f"  {'OK' if os.path.exists(os.path.join(DRIVE_ROOT, s)) else 'MISSING'} {s}")

  Added to LD_LIBRARY_PATH: /usr/local/lib/python3.12/dist-packages/nvidia/nvjitlink/lib
  Added to LD_LIBRARY_PATH: /usr/local/lib/python3.12/dist-packages/nvidia/cuda_runtime/lib
  MISSING training/scripts/stem_rewards.py
  MISSING training/scripts/verify_answers.py
  MISSING training/scripts/sort_curriculum.py


In [2]:
# ============================================================
# Configuration — GSPO-aligned (arXiv 2507.18071)
# SPEED-OPTIMIZED CONFIG — G4 RTX PRO 6000 (96GB)
#
# Strategy: thinking=True + ThinkingBudgetProcessor(1536) + Cold-Start SFT
#
# Key corrections from literature review (2026-03-23):
# 1. ThinkingBudgetProcessor: LogitsProcessor forces </think> after budget
#    (solves TRL unbounded thinking — vLLM #15418, DeepSeek-R1, DAPO)
# 2. Cold-Start SFT Phase A: 200 steps on rejection-sampled correct solutions
#    (DeepSeek-R1 approach — raises baseline from ~5% to 15-30%)
# 3. Cold-Start SFT Phase B: 100 steps on Socratic dialogues
#    (establishes Socratic behavioral prior before RL)
# 4. LoRA r=16 optimal for RL (Tina paper arXiv 2504.15777)
#    DO NOT increase to r=64 — it DEGRADES RL performance
# 5. LR=5e-7 conservative (sparse reward, prevents instability)
# 6. ReDit dithering=0.05 WITHOUT zero-variance masking (mutually exclusive)
# 7. dropout=0.0 (no regularization needed with RL's inherent exploration)
#
# CRITICAL FIXES from post-mortem (2026-03-23):
# 8. BETA 0.0→0.04: KL penalty preserves instruction-following
#    (DeepSeek-R1 uses 0.001; DAPO beta=0 only valid for base models)
# 9. System prompt MATCHES deployment: Socratic in training = Socratic at inference
#    (Llama 2 Ghost Attention — arXiv 2307.09288)
# 10. Socratic reward RE-ADDED with weight 0.45
#     (GDPO normalization + MO-GRPO variance-aware weighting)
# 11. REWARD_WEIGHTS [0.85,0.15]→[0.4,0.15,0.45]: correctness, format, socratic
#
# Speed optimizations (2026-03-19):
# 12. MAX_COMPLETION->4096 (budget=2048 + 2048 answer; +33% headroom)
# 13. GRAD_ACCUM 32→16 (effective batch 16 — standard for GRPO, −50% gen/round)
# 14. steps_per_generation 8→16 (better GPU utilization, IS corrects staleness)
# 15. SAVE_STEPS 50→100 (fewer Drive flushes)
# ============================================================
BASE_MODEL = "Qwen/Qwen3.5-9B"
OUTPUT_DIR = "/content/drive/MyDrive/MITS/checkpoints/gspo_qwen3.5_9b_v2"

# Memory optimization: MUST be set before any torch allocation
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ---- Auto-detect GPU ----
import torch
_gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
_gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3) if torch.cuda.is_available() else 0

# thinking=True + ThinkingBudgetProcessor: completions ~2048-4096 tok
if _gpu_mem_gb >= 90:
    # G4 RTX PRO 6000 / H100 — G=8 with thinking
    GPU_TYPE = f"{_gpu_name} ({_gpu_mem_gb:.0f}GB)"
    G = 8
    BATCH_SIZE = 1                     # BS=1 (BS=2 OOMs on backward); GA=16 for speed
    MAX_COMPLETION = 4096               # budget=2048 + answer=2048
    GRADIENT_ACCUMULATION_STEPS = 16    # effective batch = 1 * 16 = 16 prompts (128 completions)
elif _gpu_mem_gb >= 70:
    # A100 80GB
    GPU_TYPE = f"{_gpu_name} ({_gpu_mem_gb:.0f}GB)"
    G = 8
    BATCH_SIZE = 1  # BS=2 OOMs on backward with G=8
    MAX_COMPLETION = 4096
    GRADIENT_ACCUMULATION_STEPS = 16
elif _gpu_mem_gb >= 40:
    # A100 40GB — smaller G
    GPU_TYPE = f"{_gpu_name} ({_gpu_mem_gb:.0f}GB)"
    G = 4
    BATCH_SIZE = 1
    MAX_COMPLETION = 4096
    GRADIENT_ACCUMULATION_STEPS = 16
else:
    raise RuntimeError(f"GPU has only {_gpu_mem_gb:.0f}GB VRAM. Need >= 40GB.")

print(f"Detected GPU: {GPU_TYPE}")

# ---- GSPO core (arXiv 2507.18071) ----
IMPORTANCE_SAMPLING_LEVEL = "sequence"
LOSS_TYPE = "dr_grpo"
EPSILON = 3e-4
EPSILON_HIGH = 4e-4
# FIX: beta=0.04 (was 0.0). KL penalty preserves instruction-following.
# Evidence: DeepSeek-R1 uses beta=0.001; beta=0.0 only for base model reasoning (DAPO).
# For Socratic tutoring (style-constrained), non-zero beta prevents drift from
# the instruct model's cooperative behavior. (arXiv 2503.14476, 2501.12948)
BETA = 0.04
MAX_PROMPT_LENGTH = 512
MAX_GRAD_NORM = 0.1
MAX_SEQ_LENGTH = 4864       # 512 + 4096 + headroom (256)

# ---- ThinkingBudgetProcessor ----
THINKING_BUDGET = 2048      # +33% from 1536 (1024 truncated 49% in v1)

# ---- Cold-Start SFT Phase A: Math capability (DeepSeek-R1) ----
COLD_START_SFT_STEPS = 200          # SFT steps on rejection-sampled solutions
COLD_START_NUM_PROBLEMS = 500       # easy problems to generate solutions for
COLD_START_MIN_CORRECT = 100        # minimum correct solutions needed

# ---- Cold-Start SFT Phase B: Socratic style (NEW) ----
# Establishes Socratic behavioral prior before RL.
# Uses dialogs.jsonl (3,875 Socratic dialogues) with the deployment system prompt.
# Evidence: DeepSeek-R1 Section 3.2 — "without cold start, model has no behavioral
# prior for desired output format, and RL may not converge."
COLD_START_SOCRATIC_STEPS = 100     # SFT steps on Socratic dialogues
COLD_START_SOCRATIC_MAX_SAMPLES = 500  # max dialogues to use
SOCRATIC_DIALOGS_PATH = "/content/drive/MyDrive/training/data/dialogs.jsonl"

# ---- Stage 1 optimizer: AdamW (conservative LR for sparse signal) ----
LEARNING_RATE = 5e-7        # conservative — sparse correctness with noisy reward
WEIGHT_DECAY = 0.1
ADAM_BETA2 = 0.99

# ---- Stage 2 optimizer: Lion (arXiv 2302.06675) ----
STAGE2_OPTIMIZER = "lion_8bit"  # requires working bitsandbytes; fallback: "adamw_torch_fused"
STAGE2_LEARNING_RATE = 2e-7         # 2.5x smaller than stage 1 (Lion convention)
STAGE2_WEIGHT_DECAY = 0.3

# ---- ReDit: Reward Dithering (arXiv 2506.18631) ----
# sigma=0.05 creates continuous gradient landscape for better convergence.
# NOTE: Do NOT combine with zero-variance masking — they are mutually exclusive.
# ReDit adds noise that breaks exact zero-variance, defeating the masking logic.
DITHERING_SIGMA = 0.05

# LoRA — r=16 optimal for RL (Tina paper arXiv 2504.15777)
# DO NOT increase to r=64 — Tina shows r=64 UNDERPERFORMS r=16 (46.95% vs 48.92%)
# LoRA Safety paper (arXiv 2507.17075): r=1 is sufficient for style/behavior change
# Key: target_modules coverage matters MORE than rank (LoRA Without Regret, Schulman 2025)
LORA_R = 16
LORA_ALPHA = 32             # 2x ratio
LORA_DROPOUT = 0.0          # no dropout for RL (exploration is inherent)
TARGET_MODULES = [
    # Standard attention (8 layers)
    "q_proj", "k_proj", "v_proj", "o_proj",
    # GatedDeltaNet linear attention (24 layers) — Qwen3.5 naming
    "in_proj_qkv",   # fused Q/K/V (largest projection)
    "in_proj_z",      # gate/decay signal
    "in_proj_b",      # beta: write strength per head
    "in_proj_a",      # alpha: decay gate per head
    "out_proj",        # output projection
    # MLP (all 32 layers)
    "gate_proj", "up_proj", "down_proj",
]

# Training — speed-optimized
STEPS_PER_GENERATION = 24   # 24 gradient steps per gen round (amortize expensive generation)
LOGGING_STEPS = 5
SAVE_STEPS = 100              # checkpoint every 100 steps (3 recovery points in 300 steps)

# GDPO-style decoupled reward weights (arXiv 2601.05242)
# FIX: Socratic reward RE-ADDED with weight 0.45 (was removed entirely).
# With GDPO normalize_then_sum, each reward is variance-normalized before weighting,
# so 0.45 gives genuine 45% influence on gradient direction.
# Evidence: MO-GRPO (arXiv 2509.22047) Theorem 1 — variance-dominance bias.
# [correctness, format, socratic]
REWARD_WEIGHTS = [0.4, 0.15, 0.45]

# Curriculum learning (GRPO-LEAD, arXiv 2504.09696)
CURRICULUM_CONFIG = {
    "stage1_steps": 300,        # easy+medium only
    "stage2_steps": 500,        # all difficulties with reweighting
    "difficulty_weights": {
        "easy": 0.4,
        "medium": 1.0,
        "hard": 1.5,
    },
}
TOTAL_STEPS = CURRICULUM_CONFIG["stage1_steps"] + CURRICULUM_CONFIG["stage2_steps"]

DOMAINS = ["math", "physics", "chemistry", "biology", "cs"]

effective_batch = BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
gen_rounds = TOTAL_STEPS // STEPS_PER_GENERATION
total_completions = TOTAL_STEPS * BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS * G
lora_params_m = 2 * LORA_R * 4096 * len(TARGET_MODULES) * 2 / 1e6
print(f"Base model: {BASE_MODEL}")
print(f"G={G}, max_completion={MAX_COMPLETION}, max_seq={MAX_SEQ_LENGTH}")
print(f"Batch: {BATCH_SIZE} x {GRADIENT_ACCUMULATION_STEPS} = {effective_batch} effective prompts/step")
print(f"LoRA: r={LORA_R}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT} (~{lora_params_m:.1f}M params)")
print(f"Curriculum: {CURRICULUM_CONFIG['stage1_steps']}+{CURRICULUM_CONFIG['stage2_steps']} = {TOTAL_STEPS} steps")
print(f"Generation: {gen_rounds} rounds x {STEPS_PER_GENERATION} steps/round, {total_completions:,} total completions")
print(f"GDPO weights: {REWARD_WEIGHTS}, Loss: {LOSS_TYPE}, Beta: {BETA}")
print(f"ReDit dithering: sigma={DITHERING_SIGMA}")
print(f"ThinkingBudget: {THINKING_BUDGET} tokens (forces </think> after budget)")
print(f"Cold-Start: Phase A={COLD_START_SFT_STEPS} steps (math), Phase B={COLD_START_SOCRATIC_STEPS} steps (Socratic)")
print(f"LR: {LEARNING_RATE} (Stage 1), {STAGE2_LEARNING_RATE} (Stage 2)")
print(f"Save every {SAVE_STEPS} steps")
_est_sec = (TOTAL_STEPS + COLD_START_SFT_STEPS + COLD_START_SOCRATIC_STEPS) * 45
print(f"Estimated total time: ~{_est_sec/3600:.1f}h (SFT {COLD_START_SFT_STEPS}+{COLD_START_SOCRATIC_STEPS} + GSPO {TOTAL_STEPS} steps)")

Detected GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition (95GB)
Base model: Qwen/Qwen3.5-9B
G=8, max_completion=4096, max_seq=4864
Batch: 1 x 16 = 16 effective prompts/step
LoRA: r=16, alpha=32, dropout=0.0 (~3.1M params)
Curriculum: 300+500 = 800 steps
Generation: 33 rounds x 24 steps/round, 102,400 total completions
GDPO weights: [0.4, 0.15, 0.45], Loss: dr_grpo, Beta: 0.04
ReDit dithering: sigma=0.05
ThinkingBudget: 2048 tokens (forces </think> after budget)
Cold-Start: Phase A=200 steps (math), Phase B=100 steps (Socratic)
LR: 5e-07 (Stage 1), 2e-07 (Stage 2)
Save every 100 steps
Estimated total time: ~13.8h (SFT 200+100 + GSPO 800 steps)


In [3]:
# ============================================================
# Mount Google Drive
# ============================================================
import json
import os
import sys
from collections import Counter, defaultdict

# Try mounting Drive; fall back to local
DRIVE_MOUNTED = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_MOUNTED = True
    print("Google Drive mounted successfully")
except Exception as e:
    print(f"Drive mount failed ({e}), using local storage")
    OUTPUT_DIR = "/content/checkpoints/gspo_qwen3.5_9b"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

Mounted at /content/drive
Google Drive mounted successfully
Output directory: /content/drive/MyDrive/MITS/checkpoints/gspo_qwen3.5_9b_v2


In [4]:
# ============================================================
# Loguru — structured logging with timestamps + file persistence
# Replaces print() for: timestamps, log levels, Drive file backup
# ============================================================
from loguru import logger
import sys as _sys

# Remove default handler (no duplicate output)
logger.remove()

# Console: colored, timestamped, all levels
logger.add(
    _sys.stderr,
    format="<green>{time:HH:mm:ss}</green> | <level>{level: <8}</level> | {message}",
    level="DEBUG",
    colorize=True,
)

# File: persistent log on Drive (survives Colab disconnects!)
_log_path = os.path.join(OUTPUT_DIR, "gspo_training.log")
logger.add(
    _log_path,
    format="{time:YYYY-MM-DD HH:mm:ss} | {level: <8} | {message}",
    level="DEBUG",
    rotation="50 MB",
    retention="7 days",
)

logger.success(f"Loguru ready. File log: {_log_path}")

12:56:40 | SUCCESS  | Loguru ready. File log: /content/drive/MyDrive/MITS/checkpoints/gspo_qwen3.5_9b_v2/gspo_training.log


In [5]:
# ============================================================
# Load RL problems: Drive JSONL → HF "rl" config → HF "gspo" fallback
# ============================================================

import json
from datasets import load_dataset
from pathlib import Path

RL_DATA_PATH = "/content/drive/MyDrive/training/data/rl_combined.jsonl"

def load_rl_problems():
    """Load RL dataset with fallback chain:
    1. Local JSONL from Google Drive (prepared by prepare_rl_dataset.py)
    2. HuggingFace "rl" config (future upload)
    3. HuggingFace "gspo" config (original, for backwards compat)
    """
    # Try Drive JSONL first
    if os.path.exists(RL_DATA_PATH):
        logger.info(f"Loading RL data from Drive: {RL_DATA_PATH}")
        problems = []
        with open(RL_DATA_PATH, "r", encoding="utf-8") as f:
            for line in f:
                problems.append(json.loads(line))
        logger.success(f"Loaded {len(problems)} problems from Drive JSONL")
        return problems

    # Try HF "rl" config
    try:
        logger.info("Trying HF dataset config 'rl'...")
        hf_ds = load_dataset("Siesher/mits-stem-training-data", "rl")
        problems = [dict(r) for r in hf_ds["train"]]
        if "test" in hf_ds:
            problems += [dict(r) for r in hf_ds["test"]]
        logger.success(f"Loaded {len(problems)} problems from HF 'rl' config")
        return problems
    except Exception:
        pass

    # Fallback: HF "gspo" config
    logger.info("Falling back to HF 'gspo' config...")
    hf_ds = load_dataset("Siesher/mits-stem-training-data", "gspo")
    problems = [dict(r) for r in hf_ds["train"]] + [dict(r) for r in hf_ds["test"]]
    logger.success(f"Loaded {len(problems)} problems from HF 'gspo' config (fallback)")
    return problems


problems = load_rl_problems()

# Filter to verifiable problems only
verifiable_problems = [
    p for p in problems
    if p.get("type", "verifiable") == "verifiable"
    and p.get("answer_type", "numeric") != "conceptual"
]
conceptual_count = len(problems) - len(verifiable_problems)

logger.success(f"\nTotal loaded: {len(problems)}")
logger.info(f"  Verifiable (used for training): {len(verifiable_problems)}")
if conceptual_count > 0:
    logger.debug(f"  Conceptual (skipped — no verifiable reward signal): {conceptual_count}")

# Stats
from collections import Counter
domain_counts = Counter(p.get("domain", "unknown") for p in verifiable_problems)
source_counts = Counter(p.get("source", "unknown") for p in verifiable_problems)
type_counts = Counter(p.get("answer_type", "unknown") for p in verifiable_problems)

logger.info(f"\nBy domain:")
for domain, count in sorted(domain_counts.items()):
    logger.info(f"  {domain}: {count}")
logger.info(f"\nBy source:")
for source, count in sorted(source_counts.items()):
    logger.info(f"  {source}: {count}")
logger.info(f"\nBy answer type:")
for atype, count in sorted(type_counts.items()):
    logger.info(f"  {atype}: {count}")

12:56:41 | INFO     | Loading RL data from Drive: /content/drive/MyDrive/training/data/rl_combined.jsonl
12:56:42 | SUCCESS  | Loaded 14203 problems from Drive JSONL
12:56:42 | SUCCESS  | 
Total loaded: 14203
12:56:42 | INFO     |   Verifiable (used for training): 14203
12:56:42 | INFO     | 
By domain:
12:56:42 | INFO     |   biology: 307
12:56:42 | INFO     |   chemistry: 162
12:56:42 | INFO     |   cs: 258
12:56:42 | INFO     |   math: 12931
12:56:42 | INFO     |   physics: 545
12:56:42 | INFO     | 
By source:
12:56:42 | INFO     |   current_filtered: 91
12:56:42 | INFO     |   gsm8k: 7400
12:56:42 | INFO     |   math_hendrycks: 4985
12:56:42 | INFO     |   olympiad_bench: 232
12:56:42 | INFO     |   rummlu: 1495
12:56:42 | INFO     | 
By answer type:
12:56:42 | INFO     |   latex_boxed: 4985
12:56:42 | INFO     |   mc_letter: 1495
12:56:42 | INFO     |   numeric: 7491
12:56:42 | INFO     |   numeric_with_unit: 232


In [6]:
# ============================================================
# Held-out evaluation split (10% stratified by domain)
# ============================================================
import random

EVAL_FRACTION = 0.10
random.seed(42)

_domain_buckets = {}
for p in verifiable_problems:
    d = p.get("domain", "math")
    _domain_buckets.setdefault(d, []).append(p)

train_problems = []
eval_problems = []
for domain, probs in _domain_buckets.items():
    random.shuffle(probs)
    n_eval = max(1, int(len(probs) * EVAL_FRACTION))
    eval_problems.extend(probs[:n_eval])
    train_problems.extend(probs[n_eval:])

logger.info(f"\nHeld-out split (seed=42):")
logger.info(f"  Train: {len(train_problems)}")
logger.info(f"  Eval:  {len(eval_problems)}")
eval_domains = Counter(p.get("domain") for p in eval_problems)
for d, c in sorted(eval_domains.items()):
    logger.info(f"    {d}: {c}")

# Use train_problems for curriculum splits below
verifiable_problems = train_problems

12:56:42 | INFO     | 
Held-out split (seed=42):
12:56:42 | INFO     |   Train: 12785
12:56:42 | INFO     |   Eval:  1418
12:56:42 | INFO     |     biology: 30
12:56:42 | INFO     |     chemistry: 16
12:56:42 | INFO     |     cs: 25
12:56:42 | INFO     |     math: 1293
12:56:42 | INFO     |     physics: 54


In [7]:
# ============================================================
# Reward functions: import from stem_rewards.py (GDPO-compatible)
# Fallback: inline definitions if import fails (Colab path issues)
# ============================================================

# Try importing GDPO reward functions from stem_rewards.py
_stem_rewards_imported = False
try:
    from training.scripts.stem_rewards import make_gdpo_reward_fns, make_gdpo_correctness_fn, make_gdpo_format_fn
    _stem_rewards_imported = True
    logger.info("Imported GDPO reward functions from training.scripts.stem_rewards")
except ImportError:
    try:
        import sys
        sys.path.insert(0, "/content/drive/MyDrive")
        from training.scripts.stem_rewards import make_gdpo_reward_fns, make_gdpo_correctness_fn, make_gdpo_format_fn
        _stem_rewards_imported = True
        logger.info("Imported GDPO reward functions (via Drive path)")
    except ImportError:
        logger.warning("WARNING: Could not import stem_rewards, defining fallback reward functions")

if not _stem_rewards_imported:
    # ---- Fallback reward functions (same logic as stem_rewards.py) ----
    import sympy
    import re

    def _extract_boxed_answer(text):
        """Extract answer from \\boxed{...} with nested brace support."""
        idx = text.rfind("\\boxed{")
        if idx != -1:
            start = idx + len("\\boxed{")
            depth, pos = 1, start
            while pos < len(text) and depth > 0:
                if text[pos] == "{": depth += 1
                elif text[pos] == "}": depth -= 1
                pos += 1
            if depth == 0:
                return text[start:pos-1].strip()
        numbers = re.findall(r"[-+]?\d*\.?\d+", text)
        return numbers[-1] if numbers else ""

    def _verify_answer(completion, answer, domain):
        """Simple verification: returns 1.0 if correct, 0.0 otherwise."""
        extracted = _extract_boxed_answer(completion)
        if not extracted or not answer:
            return 0.0
        if domain == "math":
            try:
                pred = sympy.sympify(extracted)
                gold = sympy.sympify(answer)
                if sympy.simplify(pred - gold) == 0:
                    return 1.0
                return 0.0
            except (sympy.SympifyError, TypeError, ValueError):
                return 1.0 if extracted.strip() == str(answer).strip() else 0.0
        elif domain == "physics":
            try:
                pred_num = float(re.findall(r"[-+]?\d*\.?\d+", extracted)[0])
                gold_num = float(re.findall(r"[-+]?\d*\.?\d+", str(answer))[0])
                return 1.0 if abs(pred_num - gold_num) / max(abs(gold_num), 1e-10) < 0.05 else 0.0
            except (ValueError, IndexError):
                return 1.0 if extracted.strip() == str(answer).strip() else 0.0
        else:
            return 1.0 if extracted.strip().lower() == str(answer).strip().lower() else 0.0

logger.success("Reward functions ready (GDPO-compatible, no negative penalties)")

12:56:42 | WARNING  | WARNING: Could not import stem_rewards, defining fallback reward functions
12:56:42 | SUCCESS  | Reward functions ready (GDPO-compatible, no negative penalties)


In [8]:
# ============================================================
# Classify problem difficulty + build curriculum splits
# ============================================================
from collections import Counter

try:
    from training.scripts.sort_curriculum import classify_difficulty
    logger.info("Imported classify_difficulty from training.scripts.sort_curriculum")
except ImportError:
    def classify_difficulty(example):
        """Fallback: classify by word count heuristic."""
        text = example.get("answer", "") + " " + example.get("prompt", "")
        word_count = len(text.split())
        if word_count < 50: return "easy"
        elif word_count > 150: return "hard"
        return "medium"
    logger.info("Using fallback classify_difficulty")

for p in verifiable_problems:
    p["difficulty"] = classify_difficulty(p)

difficulty_dist = Counter(p["difficulty"] for p in verifiable_problems)
logger.info(f"Difficulty distribution: {dict(difficulty_dist)}")

easy_medium_problems = [p for p in verifiable_problems if p["difficulty"] in ("easy", "medium")]
hard_problems = [p for p in verifiable_problems if p["difficulty"] == "hard"]
logger.info(f"Stage 1 (easy+medium): {len(easy_medium_problems)} problems")
logger.info(f"Stage 2 adds hard: {len(hard_problems)} problems")
logger.info(f"Total: {len(verifiable_problems)} problems")

12:56:43 | INFO     | Using fallback classify_difficulty
12:56:43 | INFO     | Difficulty distribution: {'easy': 9113, 'medium': 3627, 'hard': 45}
12:56:43 | INFO     | Stage 1 (easy+medium): 12740 problems
12:56:43 | INFO     | Stage 2 adds hard: 45 problems
12:56:43 | INFO     | Total: 12785 problems


In [9]:
# NOTE: llm_blender, weave, mergekit — only needed for merge/export cells (27-28).
# They downgrade pydantic, safetensors, protobuf — causing conflicts.
# Install ONLY when needed (before cell 27), not here.
#
print("Skipped: llm_blender/weave/mergekit install deferred to merge step")


Skipped: llm_blender/weave/mergekit install deferred to merge step


In [10]:
# ============================================================
# Verify DeltaNet CUDA kernels (installed in Cell 1)
# Not critical — model works without them (uses torch fallback)
# ============================================================
import importlib

_cc1d_ok = _fla_ok = False
try:
    _cc1d = importlib.import_module("causal_conv1d")
    _cc1d_ok = True
    print(f"causal-conv1d {_cc1d.__version__}")
except ImportError:
    print("causal-conv1d not available")

try:
    _fla = importlib.import_module("fla")
    _fla_ok = True
    print(f"flash-linear-attention {_fla.__version__}")
except ImportError:
    print("flash-linear-attention not available")

if _cc1d_ok and _fla_ok:
    print("DeltaNet fast path: READY (24/32 layers use optimized CUDA kernels)")
elif _fla_ok:
    print("DeltaNet: causal-conv1d missing, using torch fallback (slower but functional)")
else:
    print("DeltaNet: both packages missing, model will use torch fallback")


causal-conv1d not available
flash-linear-attention 0.4.2
DeltaNet: causal-conv1d missing, using torch fallback (slower but functional)


In [11]:
# ============================================================
# Load base Instruct model + fresh LoRA for GSPO
# ============================================================
import os, subprocess, sys

# HOTFIX for PIL._typing ImportError before loading Unsloth
try:
    from PIL import _typing
    if not hasattr(_typing, '_Ink'):
        print("Detected Pillow compatibility issue. Reinstalling torchvision...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "torchvision>=0.23.0", "pillow<12.0.0"])
except ImportError:
    pass

import torch
import json
import numpy as np
from datasets import Dataset
from unsloth import FastLanguageModel

# Fix: llm_blender uses removed TRANSFORMERS_CACHE (transformers v5)
# This MUST be done before importing trl, because trl imports llm_blender internally.
import transformers.utils.hub as _tf_hub
if not hasattr(_tf_hub, 'TRANSFORMERS_CACHE'):
    try:
        from huggingface_hub.constants import HF_HUB_CACHE
        _tf_hub.TRANSFORMERS_CACHE = HF_HUB_CACHE
    except ImportError:
        _tf_hub.TRANSFORMERS_CACHE = '/root/.cache/huggingface/hub'

from trl import GRPOConfig, GRPOTrainer

# ---- Load Instruct base + fresh LoRA ----
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,
    dtype=torch.bfloat16,
)

if not hasattr(tokenizer, "vocab_size") and hasattr(tokenizer, "tokenizer"):
    tokenizer = tokenizer.tokenizer

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

if hasattr(model, 'hf_device_map'):
    model.hf_device_map = {'': 0}



# ============================================================
# ThinkingBudgetProcessor — forces </think> after token budget
# Solves: TRL generation has no thinking budget, model enters
# unbounded thinking loop (100% clipped at any MAX_COMPLETION).
# Reference: DeepSeek-R1, DAPO, vLLM issue #15418
# ============================================================
from transformers import LogitsProcessor

class ThinkingBudgetProcessor(LogitsProcessor):
    """Forces </think> token after a thinking budget is reached.

    Behavior:
    - At 90% of budget: boost </think> logits by +5.0 (soft nudge)
    - At 100% of budget: force </think> by setting all other logits to -inf
    - Tracks per-sequence token count since <think> tag
    - Resets state on each new generate() call via _reset()

    Args:
        think_end_token_id: Token ID for </think>
        thinking_budget: Max tokens allowed in thinking phase
        think_start_token_id: Token ID for <think> (to detect thinking start)
    """

    def __init__(self, think_end_token_id: int, thinking_budget: int = 1536,
                 think_start_token_id: int = None):
        self.think_end_token_id = think_end_token_id
        self.thinking_budget = thinking_budget
        self.think_start_token_id = think_start_token_id
        self.soft_nudge_threshold = int(thinking_budget * 0.9)
        # Per-sequence state: tokens generated since <think> tag
        self._thinking_tokens = None
        self._in_thinking = None

    def _reset(self, batch_size: int):
        """Reset state for a new generate() call."""
        self._thinking_tokens = torch.zeros(batch_size, dtype=torch.long)
        # Assume all sequences start in thinking mode (prompt ends with <think>)
        self._in_thinking = torch.ones(batch_size, dtype=torch.bool)

    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor) -> torch.FloatTensor:
        batch_size = input_ids.shape[0]

        # Initialize on first call
        if self._thinking_tokens is None or self._thinking_tokens.shape[0] != batch_size:
            self._reset(batch_size)

        for i in range(batch_size):
            if not self._in_thinking[i]:
                continue

            # Check if </think> was already generated
            last_token = input_ids[i, -1].item()
            if last_token == self.think_end_token_id:
                self._in_thinking[i] = False
                continue

            self._thinking_tokens[i] += 1
            tok_count = self._thinking_tokens[i].item()

            if tok_count >= self.thinking_budget:
                # HARD FORCE: only allow </think>
                scores[i, :] = float('-inf')
                scores[i, self.think_end_token_id] = 0.0
            elif tok_count >= self.soft_nudge_threshold:
                # SOFT NUDGE: boost </think> probability
                scores[i, self.think_end_token_id] += 5.0

        return scores


# Find </think> token ID
_think_end_token = "</think>"
_think_end_ids = tokenizer.encode(_think_end_token, add_special_tokens=False)
if len(_think_end_ids) == 1:
    THINK_END_TOKEN_ID = _think_end_ids[0]
else:
    # Fallback: search vocab
    THINK_END_TOKEN_ID = tokenizer.convert_tokens_to_ids(_think_end_token)
    if THINK_END_TOKEN_ID == tokenizer.unk_token_id:
        raise ValueError(f"Cannot find </think> token ID in tokenizer vocab!")

_think_start_ids = tokenizer.encode("<think>", add_special_tokens=False)
THINK_START_TOKEN_ID = _think_start_ids[0] if len(_think_start_ids) == 1 else None

logger.info(f"ThinkingBudgetProcessor: </think> token_id={THINK_END_TOKEN_ID}, budget={THINKING_BUDGET}")
logger.info(f"  Soft nudge at {int(THINKING_BUDGET * 0.9)} tokens, hard force at {THINKING_BUDGET} tokens")

# Create global processor instance
thinking_processor = ThinkingBudgetProcessor(
    think_end_token_id=THINK_END_TOKEN_ID,
    thinking_budget=THINKING_BUDGET,
    think_start_token_id=THINK_START_TOKEN_ID,
)

# ---- Monkey-patch model.generate to inject ThinkingBudgetProcessor ----
_original_generate = model.generate.__func__ if hasattr(model.generate, '__func__') else model.generate

def _generate_with_thinking_budget(self, *args, **kwargs):
    """Wrapper that injects ThinkingBudgetProcessor into every generate() call."""
    # Reset processor state for new generation
    # (batch_size will be auto-detected on first __call__)
    thinking_processor._thinking_tokens = None
    thinking_processor._in_thinking = None

    # Inject processor into logits_processor list
    existing_processors = kwargs.get('logits_processor', None) or []
    if not isinstance(existing_processors, list):
        existing_processors = list(existing_processors)
    # Avoid double-adding
    if not any(isinstance(p, ThinkingBudgetProcessor) for p in existing_processors):
        existing_processors.append(thinking_processor)
    kwargs['logits_processor'] = existing_processors

    return _original_generate(self, *args, **kwargs)

import types
model.generate = types.MethodType(_generate_with_thinking_budget, model)
logger.debug("[patch] model.generate monkey-patched with ThinkingBudgetProcessor")


# ============================================================
# Type-aware system prompts — SOCRATIC (matches deployment)
# FIX: Previous prompts said "solve and write \boxed{}" — direct opposite
# of Socratic deployment. Now: <think> solves with \boxed{}, visible
# output is Socratic. Correctness reward reads \boxed{} from <think>.
# Evidence: Llama 2 Ghost Attention (arXiv 2307.09288) — system prompt
# must be present during training or model forgets it at inference.
# ============================================================
SYSTEM_PROMPT_CALC = (
    "Ты — сократический репетитор по STEM.\n\n"
    "ИНСТРУКЦИИ:\n"
    "1. В секции размышлений (<think>): реши задачу полностью и ОБЯЗАТЕЛЬНО "
    "запиши финальный ответ в формате \\boxed{ответ}. Пример: \\boxed{42}.\n"
    "2. В видимом ответе студенту (после </think>):\n"
    "   - НИКОГДА не давай готовый ответ и не используй \\boxed{}\n"
    "   - Задай 1-2 наводящих вопроса, которые подведут к решению\n"
    "   - Дай подсказку про первый шаг, не раскрывая дальнейших\n"
    "   - Используй LaTeX ($...$) для математических выражений\n"
    "   - Будь дружелюбным и поддерживающим\n"
    "   - Отвечай на русском языке"
)

SYSTEM_PROMPT_MC = (
    "Ты — сократический репетитор по STEM.\n\n"
    "ИНСТРУКЦИИ:\n"
    "1. В секции размышлений (<think>): проанализируй все варианты "
    "и определи правильный ответ (A, B, C или D).\n"
    "2. В видимом ответе студенту (после </think>):\n"
    "   - НИКОГДА не называй правильный вариант напрямую\n"
    "   - Задай вопрос, который поможет студенту исключить неправильные варианты\n"
    "   - Дай подсказку про ключевое отличие между вариантами\n"
    "   - Отвечай на русском языке"
)

# ---- Task Generation system prompt (matches deployment TASK_GENERATOR_MODE_SYSTEM) ----
SYSTEM_PROMPT_TASKGEN = (
    "Ты — эксперт по созданию образовательных задач по математике и STEM.\n\n"
    "ТВОЯ ЗАДАЧА:\n"
    "Пользователь описывает, какие задачи ему нужны (тему, сложность, количество).\n"
    "Ты генерируешь задачи с пошаговыми решениями.\n\n"
    "ФОРМАТ ОТВЕТА:\n"
    "Для каждой задачи выведи:\n"
    "1. Условие с $LaTeX$ формулами\n"
    "2. Пошаговое решение\n"
    "3. Финальный ответ в формате \\boxed{ответ}\n\n"
    "ВАЖНО: Все тексты на РУССКОМ языке!"
)

# Legacy alias for curriculum/reward code
SYSTEM_PROMPT = SYSTEM_PROMPT_CALC

# Multi-mode prompt routing: 70% Socratic, 20% TaskGen, 10% MC-Socratic
# This prevents catastrophic forgetting of task generation capability.
# Evidence: InstructGPT (arXiv 2203.02155) — training distribution must
# cover all deployment scenarios to avoid alignment tax on uncovered modes.
import random as _prompt_random
_prompt_random.seed(42)

PROMPT_MODE_WEIGHTS = {
    "socratic": 0.70,    # guided learning mode
    "taskgen": 0.20,     # task generation mode
    "mc_socratic": 0.10, # MC questions with Socratic guidance
}

def get_system_prompt(problem):
    """Return system prompt based on answer_type + stochastic mode mixing.

    For non-MC problems:
      - 78% chance: Socratic tutoring prompt (learn to guide students)
      - 22% chance: Task generation prompt (preserve task gen capability)
    For MC problems:
      - Always Socratic MC prompt (too few MC to split further)

    The stochastic mixing prevents the model from losing task generation
    ability while primarily optimizing for Socratic tutoring.
    """
    if problem.get("answer_type") == "mc_letter":
        return SYSTEM_PROMPT_MC

    # Stochastic mode mixing for non-MC problems
    prompt_text = problem.get("prompt", "")
    # Meta-prompts ("Сгенерируй...") always get task generation prompt
    if prompt_text.startswith("Сгенерируй") or prompt_text.startswith("Generate"):
        return SYSTEM_PROMPT_TASKGEN

    # For real problems: probabilistic mixing
    r = _prompt_random.random()
    if r < 0.78:
        return SYSTEM_PROMPT_CALC  # Socratic
    else:
        return SYSTEM_PROMPT_TASKGEN  # Task generation


# ============================================================
# GDPO-style reward functions (NO zero-variance masking — uses ReDit instead)
# ============================================================

def format_problems_as_dataset(problem_list):
    """Format problems into a Dataset for GRPOTrainer.
    Routes each problem to the correct system prompt by answer_type.
    """
    formatted = []
    for p in problem_list:
        messages = [
            {"role": "system", "content": get_system_prompt(p)},
            {"role": "user", "content": p["prompt"]},
        ]
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
            enable_thinking=True,   # ON: model needs CoT for STEM; ThinkingBudgetProcessor caps thinking
        )
        formatted.append({"prompt": prompt})
    return Dataset.from_list(formatted)


# Build prompt-to-problem lookup (shared by reward functions)
# Registers each problem with its type-appropriate system prompt
_prompt_to_problem = {}
for p in verifiable_problems:
    messages = [
        {"role": "system", "content": get_system_prompt(p)},
        {"role": "user", "content": p["prompt"]},
    ]
    formatted = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
        enable_thinking=True,   # ON: model needs CoT for STEM; ThinkingBudgetProcessor caps thinking
    )
    _prompt_to_problem[formatted.strip()] = p


if _stem_rewards_imported:
    # Use proper GDPO functions from stem_rewards.py
    # These now handle MC/calc routing + ReDit dithering internally
    _gdpo_fns = make_gdpo_reward_fns(
        verifiable_problems, tokenizer, SYSTEM_PROMPT_CALC,
        dithering_sigma=DITHERING_SIGMA,  # ReDit (arXiv 2506.18631)
    )
    _base_correctness_fn = _gdpo_fns[0]
    _base_format_fn = _gdpo_fns[1]
    _base_socratic_fn = _gdpo_fns[2] if len(_gdpo_fns) > 2 else None
    logger.info(f"  ReDit dithering: sigma={DITHERING_SIGMA}")
else:
    # Fallback: build correctness fn using inline _verify_answer
    def _base_correctness_fn(completions, prompts=None, **kwargs):
        if prompts is None:
            prompts = [""] * len(completions)
        rewards = []
        for prompt_text, completion_text in zip(prompts, completions):
            problem = _prompt_to_problem.get(prompt_text.strip())
            if problem is None:
                rewards.append(0.0)
                continue
            domain = problem.get("domain", "math")
            answer = problem.get("answer", "")
            answer_type = problem.get("answer_type", "numeric")

            if answer_type == "mc_letter":
                import re
                mc_match = re.search(r'\b([A-DА-Г])\b', _extract_boxed_answer(completion_text) or completion_text[-20:])
                rewards.append(1.0 if mc_match and mc_match.group(1).upper() == answer.upper() else 0.0)
            else:
                rewards.append(_verify_answer(completion_text, answer, domain))

        # ReDit dithering
        if DITHERING_SIGMA > 0:
            noise = np.random.normal(0.0, DITHERING_SIGMA, size=len(rewards))
            rewards = [r + n for r, n in zip(rewards, noise)]

        return rewards

    def _base_format_fn(completions, prompts=None, **kwargs):
        rewards = []
        for i, text in enumerate(completions):
            text = text if isinstance(text, str) else str(text)
            answer_type = "numeric"
            if prompts and i < len(prompts):
                problem = _prompt_to_problem.get(prompts[i].strip())
                if problem:
                    answer_type = problem.get("answer_type", "numeric")

            score = 0.0
            if answer_type == "mc_letter":
                import re
                if re.search(r'(?:answer|ответ)\s*[:=]\s*[A-DА-Г]', text, re.IGNORECASE):
                    score += 0.5
                step_markers = ["because", "therefore", "потому что", "так как", "следовательно"]
                if any(m in text.lower() for m in step_markers):
                    score += 0.3
            else:
                if "\\boxed{" in text:
                    score += 0.5
                step_markers = ["step", "therefore", "thus", "hence", "because",
                                "шаг", "следовательно", "значит", "потому что",
                                "так как", "далее", "подставим", "найдём"]
                if any(m in text.lower() for m in step_markers):
                    score += 0.3
            word_count = len(text.split())
            if 50 < word_count < 800:
                score += 0.2
            rewards.append(min(1.0, score))
        return rewards


# ---- Difficulty-aware correctness reward (GRPO-LEAD) ----
# This is the FINAL correctness reward used in training.
# NO zero-variance masking — ReDit dithering handles gradient landscape.
DIFFICULTY_WEIGHTS = CURRICULUM_CONFIG["difficulty_weights"]

def difficulty_weighted_correctness_fn(completions, prompts=None, **kwargs):
    """Correctness reward x difficulty weight (GRPO-LEAD reweighting).

    Uses ReDit dithering (from _base_correctness_fn) for continuous gradients.
    No zero-variance masking needed — ReDit creates gradient signal even in
    all-wrong groups.
    """
    base_rewards = _base_correctness_fn(completions, prompts=prompts, **kwargs)
    if prompts is None:
        return base_rewards
    weighted = []
    for reward, prompt_text in zip(base_rewards, prompts):
        problem = _prompt_to_problem.get(prompt_text.strip())
        if problem is not None:
            diff = problem.get("difficulty", "medium")
            weight = DIFFICULTY_WEIGHTS.get(diff, 1.0)
            weighted.append(reward * weight)
        else:
            weighted.append(reward)
    return weighted


# Fallback socratic reward (if stem_rewards import failed or no socratic fn)
if not _stem_rewards_imported or '_base_socratic_fn' not in dir() or _base_socratic_fn is None:
    def _base_socratic_fn(completions, prompts=None, **kwargs):
        """Score Socratic tutoring quality for each completion.

        Prompt-aware: returns neutral score (0.5) for TaskGen prompts,
        so Socratic reward doesn't penalize correct task generation.
        """
        import re as _re
        scores = []
        for idx, c in enumerate(completions):
            # Check if this prompt uses TaskGen system prompt
            # If so, Socratic scoring is irrelevant — return neutral 0.5
            if prompts and idx < len(prompts):
                prompt_text = prompts[idx] if isinstance(prompts[idx], str) else str(prompts[idx])
                if "эксперт по созданию образовательных задач" in prompt_text:
                    scores.append(0.5)  # neutral — GDPO normalization ignores constant
                    continue

            text = c if isinstance(c, str) else str(c)
            if "</think>" in text:
                visible = text.split("</think>")[-1].strip()
            else:
                visible = text.strip()
            if not visible:
                scores.append(0.0)
                continue
            score = 0.0
            qcount = visible.count("?")
            if qcount >= 2: score += 0.5
            elif qcount >= 1: score += 0.3
            patterns = ["как ты думаешь", "попробуй", "подумай", "вспомни",
                       "давай разберём", "а если", "что будет если", "почему"]
            hits = sum(1 for p in patterns if _re.search(p, visible, _re.IGNORECASE))
            if hits >= 2: score += 0.4
            elif hits >= 1: score += 0.2
            telling = [r"ответ\s*[:=]", "правильный ответ", r"решение\s*[:=]"]
            thits = sum(1 for p in telling if _re.search(p, visible, _re.IGNORECASE))
            if thits > 0: score -= 0.3 * thits
            scores.append(max(0.0, min(1.0, score)))
        return scores
    logger.info("  Using fallback Socratic reward function")

# Stats
mc_count = sum(1 for p in verifiable_problems if p.get("answer_type") == "mc_letter")
calc_count = len(verifiable_problems) - mc_count
logger.success(f"\nModel and GDPO reward functions ready for curriculum GSPO training")
logger.info(f"  Loss type: {LOSS_TYPE}")
logger.info(f"  ReDit dithering: sigma={DITHERING_SIGMA} (no zero-variance masking)")
logger.info(f"  ThinkingBudgetProcessor: {THINKING_BUDGET} tokens")
logger.info(f"  Calc problems (\\boxed{{}}): {calc_count}")
logger.info(f"  MC problems (A/B/C/D): {mc_count}")
logger.info(f"  Difficulty weights: {DIFFICULTY_WEIGHTS}")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.10: Fast Qwen3_5 patching. Transformers: 5.3.0.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 12.0. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/760 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/12.8M [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

Unsloth: Making `model.base_model.model.model.language_model` require gradients


12:58:13 | INFO     | ThinkingBudgetProcessor: </think> token_id=248069, budget=2048
12:58:13 | INFO     |   Soft nudge at 1843 tokens, hard force at 2048 tokens
12:58:13 | DEBUG    | [patch] model.generate monkey-patched with ThinkingBudgetProcessor
12:58:14 | INFO     |   Using fallback Socratic reward function
12:58:14 | SUCCESS  | 
Model and GDPO reward functions ready for curriculum GSPO training
12:58:14 | INFO     |   Loss type: dr_grpo
12:58:14 | INFO     |   ReDit dithering: sigma=0.05 (no zero-variance masking)
12:58:14 | INFO     |   ThinkingBudgetProcessor: 2048 tokens
12:58:14 | INFO     |   Calc problems (\boxed{}): 11435
12:58:14 | INFO     |   MC problems (A/B/C/D): 1350
12:58:14 | INFO     |   Difficulty weights: {'easy': 0.4, 'medium': 1.0, 'hard': 1.5}


In [12]:
# ============================================================
# Qwen3.5 returns Processor (multimodal), unwrap to tokenizer
# ============================================================
if not hasattr(tokenizer, "vocab_size") and hasattr(tokenizer, "tokenizer"):
    _processor = tokenizer
    tokenizer = _processor.tokenizer
    logger.debug(f"Unwrapped Qwen3VLProcessor -> {type(tokenizer).__name__}")

# ============================================================
# Unsloth/Qwen3 compatibility patches (ALL IDEMPOTENT)
# ============================================================
import torch

# CORRECT vocab size: includes special tokens (pad, eos, im_start, etc.)
# tokenizer.vocab_size = 151643 (base only, WRONG for clamping!)
# model embedding includes all tokens
_vocab_size = model.get_input_embeddings().num_embeddings
logger.info(f"Using embedding vocab_size={_vocab_size} (not tokenizer.vocab_size={tokenizer.vocab_size})")

# ---- Patch 1: has_images ----
try:
    _gen_method = GRPOTrainer._generate_and_score_completions
    if "has_images" not in _gen_method.__globals__ or _gen_method.__globals__.get("has_images") is None:
        _gen_method.__globals__["has_images"] = False
        logger.debug("[patch 1/5] has_images=False injected")
    else:
        logger.debug(f"[patch 1/5] has_images OK")
except AttributeError:
    logger.debug("[patch 1/5] not needed")

# ---- Patch 2: batch_decode safety net ----
_P2 = "_original_batch_decode_unpatched"
if not hasattr(tokenizer, _P2):
    _orig_batch_decode = tokenizer.batch_decode

    def _safe_batch_decode(sequences, skip_special_tokens=False, **kwargs):
        sanitized = []
        for seq in sequences:
            if isinstance(seq, torch.Tensor):
                seq = seq.clamp(0, _vocab_size - 1).tolist()
            elif isinstance(seq, list):
                seq = [max(0, min(t, _vocab_size - 1)) for t in seq]
            sanitized.append(seq)
        return _orig_batch_decode(sanitized, skip_special_tokens=skip_special_tokens, **kwargs)

    tokenizer.batch_decode = _safe_batch_decode
    tokenizer._original_batch_decode_unpatched = _orig_batch_decode
    logger.debug(f"[patch 2/5] batch_decode clamped to [0, {_vocab_size})")
else:
    logger.debug("[patch 2/5] batch_decode OK")

# ---- Patch 3: embed_tokens safety net ----
_embed_layer = model.get_input_embeddings()
_P3 = "_original_embed_forward_unpatched"
if not hasattr(_embed_layer, _P3):
    _orig_embed_fwd = _embed_layer.forward

    def _safe_embed_forward(input_tensor):
        return _orig_embed_fwd(input_tensor.clamp(0, _vocab_size - 1))

    _embed_layer.forward = _safe_embed_forward
    _embed_layer._original_embed_forward_unpatched = _orig_embed_fwd
    logger.debug(f"[patch 3/5] embed_tokens clamped to [0, {_vocab_size})")
else:
    logger.debug("[patch 3/5] embed_tokens OK")

# ---- Patch 4: generation output safety net ----
_P4 = "_original_gen_score_unpatched"
_pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0

if not hasattr(GRPOTrainer, _P4):
    _orig_gen_score = GRPOTrainer._generate_and_score_completions
    _ID_KEYS = {"input_ids", "prompt_ids", "completion_ids", "labels",
                "prompt_completion_ids", "old_input_ids"}

    def _safe_gen_score(self, inputs):
        result = GRPOTrainer._original_gen_score_unpatched(self, inputs)
        if isinstance(result, dict):
            for key, val in result.items():
                if key in _ID_KEYS and isinstance(val, torch.Tensor):
                    bad = (val < 0) | (val >= _vocab_size)
                    if bad.any():
                        result[key] = torch.where(bad, _pad_id, val)
        return result

    GRPOTrainer._original_gen_score_unpatched = _orig_gen_score
    GRPOTrainer._generate_and_score_completions = _safe_gen_score
    logger.debug(f"[patch 4/5] generation: invalid IDs -> pad_token_id={_pad_id}")
else:
    logger.debug("[patch 4/5] generation OK")

logger.debug(f"\n[info] pad={tokenizer.pad_token_id}, eos={tokenizer.eos_token_id}, bos={tokenizer.bos_token_id}")

# ---- Patch 5: text-only position_ids (bypass 3D vision RoPE) ----
# Qwen3.5 is multimodal: compute_3d_position_ids() computes 3D positions
# for image/video tokens using rope_deltas. For text-only training,
# this is broken (empty rope_deltas). Complete replacement with text-only version.
# NOTE: **kwargs is required — transformers v5 may pass mm_token_type_ids, cache_position, etc.
_qwen_inner = model
while hasattr(_qwen_inner, 'model'):
    _qwen_inner = _qwen_inner.model

def _text_only_position_ids(self, input_ids=None, inputs_embeds=None,
                             image_grid_thw=None, video_grid_thw=None,
                             attention_mask=None, past_key_values=None,
                             **kwargs):
    """Text-only position IDs for Qwen3.5 (bypass 3D vision positioning).

    Accepts and ignores vision-specific args (mm_token_type_ids, etc.)
    that transformers v5+ may pass.
    """
    if inputs_embeds is not None:
        batch_size, seq_len = inputs_embeds.shape[:2]
        device = inputs_embeds.device
    else:
        batch_size, seq_len = input_ids.shape
        device = input_ids.device

    past_seen = 0
    if past_key_values is not None and hasattr(past_key_values, 'get_seq_length'):
        try:
            past_seen = past_key_values.get_seq_length()
        except Exception:
            past_seen = 0

    if attention_mask is not None and past_seen == 0:
        position_ids = attention_mask.long().cumsum(-1) - 1
        position_ids.masked_fill_(attention_mask == 0, 1)
        position_ids = position_ids.unsqueeze(0).expand(3, -1, -1)
    else:
        positions = torch.arange(past_seen, past_seen + seq_len, device=device)
        position_ids = positions.view(1, 1, -1).expand(3, batch_size, -1)

    self.rope_deltas = torch.zeros(batch_size, 1, dtype=torch.long, device=device)
    return position_ids

type(_qwen_inner).compute_3d_position_ids = _text_only_position_ids
logger.debug(f"[patch 5/5] compute_3d_position_ids replaced (text-only, **kwargs-safe) on {type(_qwen_inner).__name__}")
logger.info("All patches applied.")

12:58:14 | INFO     | Using embedding vocab_size=248320 (not tokenizer.vocab_size=248044)
12:58:14 | DEBUG    | [patch 1/5] has_images=False injected
12:58:14 | DEBUG    | [patch 2/5] batch_decode clamped to [0, 248320)
12:58:14 | DEBUG    | [patch 3/5] embed_tokens clamped to [0, 248320)
12:58:14 | DEBUG    | [patch 4/5] generation: invalid IDs -> pad_token_id=248044
12:58:14 | DEBUG    | 
[info] pad=248044, eos=248046, bos=None
12:58:14 | DEBUG    | [patch 5/5] compute_3d_position_ids replaced (text-only, **kwargs-safe) on Qwen3_5Model
12:58:14 | INFO     | All patches applied.


In [13]:
logger.info(f"tokenizer.vocab_size     = {tokenizer.vocab_size}")
logger.info(f"len(tokenizer)           = {len(tokenizer)}")
_config_vocab = getattr(model.config, 'vocab_size', None) or getattr(getattr(model.config, 'text_config', None), 'vocab_size', 'N/A')
logger.info(f"model.config.vocab_size  = {_config_vocab}")
logger.info(f"embed num_embeddings     = {model.get_input_embeddings().num_embeddings}")

12:58:14 | INFO     | tokenizer.vocab_size     = 248044
12:58:14 | INFO     | len(tokenizer)           = 248077
12:58:14 | INFO     | model.config.vocab_size  = 248320
12:58:14 | INFO     | embed num_embeddings     = 248320


In [14]:
# ============================================================
# Evaluation removed — run locally:
#   python training/scripts/evaluate_stage.py --stage base --model Qwen/Qwen3.5-9B
#   python training/scripts/evaluate_stage.py --stage gspo --adapter checkpoints/gspo_qwen3.5_9b/final
# ============================================================
logger.debug("Evaluation skipped (run locally for faster iteration)")

12:58:14 | DEBUG    | Evaluation skipped (run locally for faster iteration)


In [15]:
# ============================================================
# Cold-Start SFT (DeepSeek-R1 approach)
#
# Before RL training, do 200 SFT steps on rejection-sampled correct
# solutions generated by the base model itself. This raises the
# baseline accuracy from ~5% to 15-30%, giving GRPO enough positive
# signal to learn from.
#
# Process:
# 1. Generate solutions for 500 easy problems (Batched!)
# 2. Verify correctness using stem_rewards verifier
# 3. Filter to correct solutions only (target: 100+)
# 4. SFT for 200 steps on correct solutions
#
# Skip condition: if cold_start adapter already exists on Drive
# ============================================================
import torch
import time
import os
from unsloth import FastLanguageModel
from trl import SFTConfig, SFTTrainer
from datasets import Dataset

cold_start_dir = os.path.join(OUTPUT_DIR, "cold_start")
cold_start_adapter_path = os.path.join(OUTPUT_DIR, "cold_start_adapter")
_cold_start_done = False

# Check if cold_start adapter already exists with matching LoRA rank
_cs_adapter_file = os.path.join(cold_start_adapter_path, "adapter_model.safetensors")
_cs_config_file = os.path.join(cold_start_adapter_path, "adapter_config.json")
if os.path.exists(_cs_adapter_file):
    if os.path.exists(_cs_config_file):
        import json as _json_cs
        with open(_cs_config_file, "r") as f:
            _cs_cfg = _json_cs.load(f)
        if _cs_cfg.get("r") == LORA_R and set(_cs_cfg.get("target_modules", [])) == set(TARGET_MODULES):
            _cold_start_done = True
        else:
            logger.info(f"  Cold-start adapter mismatch (r={_cs_cfg.get('r')}, modules={len(_cs_cfg.get('target_modules', []))}) — regenerating")
    else:
        _cold_start_done = True

if _cold_start_done:
    logger.info(f"{'='*60}")
    logger.debug(f"COLD-START SFT: SKIPPED (adapter found at {cold_start_adapter_path})")
    logger.info(f"{'='*60}")
    from safetensors.torch import load_file as _load_cs
    model.load_state_dict(
        _load_cs(os.path.join(cold_start_adapter_path, "adapter_model.safetensors")),
        strict=False,
    )
    logger.success("  Loaded cold-start adapter weights")
else:
    logger.info(f"{'='*60}")
    logger.info(f"COLD-START SFT: Generating solutions for {COLD_START_NUM_PROBLEMS} easy problems")
    logger.info(f"{'='*60}")

    # Step 1: Select easy problems for rejection sampling
    easy_problems = [p for p in verifiable_problems if p.get("difficulty") == "easy"]
    if len(easy_problems) < COLD_START_NUM_PROBLEMS:
        # Supplement with medium problems
        medium_problems = [p for p in verifiable_problems if p.get("difficulty") == "medium"]
        easy_problems = easy_problems + medium_problems
    cs_problems = easy_problems[:COLD_START_NUM_PROBLEMS]
    logger.info(f"  Selected {len(cs_problems)} problems for rejection sampling")

    # Step 2: Generate solutions with ThinkingBudgetProcessor
    FastLanguageModel.for_inference(model)

    # CRITICAL FOR BATCHED GENERATION: Left padding
    original_padding_side = tokenizer.padding_side
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    GEN_BATCH_SIZE = 128 # Process 48 problems at once (+167% batch, 176GB RAM confirmed, memory confirmed ok)
    correct_solutions = []
    total_generated = 0
    t0 = time.time()

    for i in range(0, len(cs_problems), GEN_BATCH_SIZE):
        batch_problems = cs_problems[i:i+GEN_BATCH_SIZE]
        prompts = []

        for p in batch_problems:
            messages = [
                {"role": "system", "content": get_system_prompt(p)},
                {"role": "user", "content": p["prompt"]},
            ]
            prompt_text = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True,
                enable_thinking=True,
            )
            prompts.append(prompt_text)

        inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_COMPLETION,
                do_sample=True,
                temperature=0.7,
                top_p=0.95,
                pad_token_id=tokenizer.pad_token_id,
            )

        for j, out_seq in enumerate(outputs):
            prompt_len = inputs["input_ids"].shape[1]
            completion = tokenizer.decode(out_seq[prompt_len:], skip_special_tokens=True)
            p = batch_problems[j]
            prompt_text = prompts[j]
            total_generated += 1

            # Verify correctness
            answer_text = None
            if _stem_rewards_imported:
                from training.scripts.verify_answers import extract_answer, verify
                answer_text = extract_answer(completion)
                truth = p.get("ground_truth", p.get("answer", ""))
                domain = p.get("domain", "math")
                answer_type = p.get("answer_type", "numeric")
                q_type = "mc" if answer_type == "mc_letter" else p.get("type", "calc")
                result = verify(answer=answer_text, truth=truth, domain=domain, question_type=q_type)
                is_correct = result.correct
            else:
                import re
                boxed = re.search(r'\\boxed\{(.+?)\}', completion)
                answer_text = boxed.group(1).strip() if boxed else ""
                truth = p.get("ground_truth", p.get("answer", ""))
                is_correct = answer_text.strip() == truth.strip()

            if is_correct:
                correct_solutions.append({
                    "text": prompt_text + completion + tokenizer.eos_token,
                    "domain": p.get("domain", "math"),
                    "difficulty": p.get("difficulty", "easy"),
                })

        # Progress logging
        elapsed = time.time() - t0
        rate = total_generated / elapsed
        logger.info(f"  [{total_generated}/{len(cs_problems)}] correct={len(correct_solutions)}/{total_generated} "
              f"({len(correct_solutions)/max(1,total_generated):.0%}), "
              f"{rate:.1f} problems/s, elapsed={elapsed:.0f}s")

        # Early stop if we have enough
        if len(correct_solutions) >= COLD_START_MIN_CORRECT * 3:
            logger.info(f"  Early stop: {len(correct_solutions)} correct solutions collected")
            break

        del outputs, inputs
        torch.cuda.empty_cache()

    # Restore original padding side
    tokenizer.padding_side = original_padding_side

    gen_time = time.time() - t0
    logger.success(f"\nRejection sampling complete: {len(correct_solutions)}/{total_generated} correct "
          f"({len(correct_solutions)/max(1,total_generated):.0%}) in {gen_time:.0f}s")

    if len(correct_solutions) < 20:
        logger.warning(f"WARNING: Only {len(correct_solutions)} correct solutions! "
              f"Cold-start SFT may not be effective. Proceeding anyway...")

    if len(correct_solutions) > 0:
        # Step 3: SFT on correct solutions
        FastLanguageModel.for_training(model)

        sft_dataset = Dataset.from_list([{"text": s["text"]} for s in correct_solutions])
        logger.info(f"\nStarting Cold-Start SFT: {len(sft_dataset)} examples, {COLD_START_SFT_STEPS} steps")

        sft_config = SFTConfig(
            output_dir=cold_start_dir,
            max_steps=COLD_START_SFT_STEPS,
            per_device_train_batch_size=4,
            gradient_accumulation_steps=4,
            optim="adamw_torch",        # avoid bitsandbytes 8-bit (libnvJitLink issue)
            learning_rate=2e-5,         # standard SFT LR
            lr_scheduler_type="cosine",
            warmup_steps=20,              # 10% of 200 SFT steps
            max_seq_length=MAX_SEQ_LENGTH,
            bf16=True,
            logging_steps=10,
            save_steps=COLD_START_SFT_STEPS,  # save only at the end
            save_total_limit=1,
            seed=42,
            report_to="none",
        )

        sft_trainer = SFTTrainer(
            model=model,
            args=sft_config,
            train_dataset=sft_dataset,
            processing_class=tokenizer,
        )

        logger.info("Training...")
        sft_result = sft_trainer.train()
        logger.success(f"Cold-Start SFT complete! Loss: {sft_result.training_loss:.4f}")

        # Save adapter
        model.save_pretrained(cold_start_adapter_path)
        tokenizer.save_pretrained(cold_start_adapter_path)
        logger.success(f"  Cold-start adapter saved to {cold_start_adapter_path}")

        del sft_trainer
        torch.cuda.empty_cache()
    else:
        logger.error("No correct solutions found — skipping Cold-Start SFT")
        FastLanguageModel.for_training(model)

# ============================================================
# Cold-Start SFT Phase B: Socratic Style (NEW)
# Establishes Socratic behavioral prior using existing dialogues.
# Evidence: DeepSeek-R1 Section 3.2 — model needs behavioral prior
# for target output format, or RL cannot converge.
# ============================================================
_socratic_sft_done = False
_socratic_adapter_path = os.path.join(OUTPUT_DIR, "cold_start_socratic_adapter")

if os.path.exists(os.path.join(_socratic_adapter_path, "adapter_model.safetensors")):
    _socratic_sft_done = True
    logger.info(f"COLD-START SOCRATIC SFT: SKIPPED (adapter found)")
    from safetensors.torch import load_file as _load_soc
    model.load_state_dict(
        _load_soc(os.path.join(_socratic_adapter_path, "adapter_model.safetensors")),
        strict=False,
    )
    logger.success("  Loaded Socratic cold-start adapter weights")
elif os.path.exists(SOCRATIC_DIALOGS_PATH):
    logger.info(f"\n{'='*60}")
    logger.info(f"COLD-START SOCRATIC SFT: {COLD_START_SOCRATIC_STEPS} steps on Socratic dialogues")
    logger.info(f"{'='*60}")

    socratic_texts = []
    with open(SOCRATIC_DIALOGS_PATH, "r", encoding="utf-8") as f:
        for line in f:
            dialog = json.loads(line)
            convs = dialog.get("conversations", [])
            if len(convs) < 3:
                continue
            messages = []
            for msg in convs:
                role = msg.get("role", "user")
                content = msg.get("content", "")
                if role == "system":
                    messages.append({"role": "system", "content": SYSTEM_PROMPT_CALC})
                else:
                    messages.append({"role": role, "content": content})
            text = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=False,
                enable_thinking=False,
            )
            socratic_texts.append({"text": text + tokenizer.eos_token})
            if len(socratic_texts) >= COLD_START_SOCRATIC_MAX_SAMPLES:
                break

    logger.info(f"  Loaded {len(socratic_texts)} Socratic dialogues")

    if len(socratic_texts) >= 20:
        FastLanguageModel.for_training(model)
        soc_dataset = Dataset.from_list(socratic_texts)

        soc_sft_config = SFTConfig(
            output_dir=os.path.join(OUTPUT_DIR, "cold_start_socratic"),
            max_steps=COLD_START_SOCRATIC_STEPS,
            per_device_train_batch_size=4,
            gradient_accumulation_steps=4,
            optim="adamw_torch",        # avoid bitsandbytes 8-bit
            learning_rate=1e-5,
            lr_scheduler_type="cosine",
            warmup_steps=10,
            max_seq_length=MAX_SEQ_LENGTH,
            bf16=True,
            logging_steps=10,
            save_steps=COLD_START_SOCRATIC_STEPS,
            save_total_limit=1,
            seed=42,
            report_to="none",
        )

        soc_trainer = SFTTrainer(
            model=model,
            args=soc_sft_config,
            train_dataset=soc_dataset,
            processing_class=tokenizer,
        )

        logger.info("Training Socratic style...")
        soc_result = soc_trainer.train()
        logger.success(f"Socratic Cold-Start SFT complete! Loss: {soc_result.training_loss:.4f}")

        model.save_pretrained(_socratic_adapter_path)
        tokenizer.save_pretrained(_socratic_adapter_path)
        logger.success(f"  Socratic adapter saved to {_socratic_adapter_path}")

        del soc_trainer
        torch.cuda.empty_cache()
    else:
        logger.warning(f"Only {len(socratic_texts)} dialogues — skipping Socratic SFT")
else:
    logger.warning(f"Socratic dialogues not found at {SOCRATIC_DIALOGS_PATH} — skipping Phase B")

logger.success(f"\nModel ready for GSPO training (Phase A: math, Phase B: Socratic)")


12:58:14 | INFO     | ============================================================
12:58:14 | DEBUG    | COLD-START SFT: SKIPPED (adapter found at /content/drive/MyDrive/MITS/checkpoints/gspo_qwen3.5_9b_v2/cold_start_adapter)
12:58:14 | INFO     | ============================================================
12:58:22 | SUCCESS  |   Loaded cold-start adapter weights
12:58:22 | INFO     | COLD-START SOCRATIC SFT: SKIPPED (adapter found)
12:58:27 | SUCCESS  |   Loaded Socratic cold-start adapter weights
12:58:27 | SUCCESS  | 
Model ready for GSPO training (Phase A: math, Phase B: Socratic)


In [16]:
# ============================================================
# PROBE: Quick parameter validation before full training
# Generates G completions for 3 problems, measures:
#   - Completion lengths (mean, min, max)
#   - Clipped ratio (% hitting MAX_COMPLETION)
#   - Peak GPU memory
#   - Time per generation
# Run this cell ONCE, check results, adjust config if needed.
# ============================================================
import torch, time, numpy as np
from unsloth import FastLanguageModel

# Sample 3 problems (easy, medium, hard)
probe_problems = []
for diff in ['easy', 'medium', 'hard']:
    for p in verifiable_problems:
        if p.get('difficulty') == diff:
            probe_problems.append(p)
            break

logger.info(f'Probe: {len(probe_problems)} problems x G={G} completions, MAX_COMPLETION={MAX_COMPLETION}')
logger.info(f'GPU before: {torch.cuda.memory_allocated()/1e9:.1f} GB allocated, {torch.cuda.max_memory_allocated()/1e9:.1f} GB peak')
torch.cuda.reset_peak_memory_stats()

FastLanguageModel.for_inference(model)

all_lengths = []
clipped = 0
total = 0

for i, p in enumerate(probe_problems):
    messages = [
        {'role': 'system', 'content': get_system_prompt(p)},
        {'role': 'user', 'content': p['prompt']},
    ]
    prompt_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
        enable_thinking=True,
    )
    inputs = tokenizer(prompt_text, return_tensors='pt').to(model.device)
    prompt_len = inputs['input_ids'].shape[1]

    t0 = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_COMPLETION,
            num_return_sequences=G,
            do_sample=True,
            temperature=1.0,
            top_p=1.0,
            pad_token_id=tokenizer.pad_token_id,
        )
    elapsed = time.time() - t0

    for seq in outputs:
        comp_len = len(seq) - prompt_len
        all_lengths.append(comp_len)
        if comp_len >= MAX_COMPLETION:
            clipped += 1
        total += 1

    lengths = [len(seq) - prompt_len for seq in outputs]
    diff = p.get('difficulty', '?')
    logger.info(f'  [{diff:>6s}] mean={np.mean(lengths):.0f}, min={min(lengths)}, max={max(lengths)}, '
          f'clipped={sum(1 for l in lengths if l >= MAX_COMPLETION)}/{G}, time={elapsed:.1f}s')

    # Decode one sample to check quality
    sample = tokenizer.decode(outputs[0][prompt_len:], skip_special_tokens=True)
    has_boxed = '\boxed{' in sample
    has_think = '</think>' in sample
    logger.info(f'         boxed={has_boxed}, think={has_think}, tokens={lengths[0]}')
    if has_think:
        parts = sample.split('</think>')
        think_len = len(tokenizer.encode(parts[0]))
        answer_len = len(tokenizer.encode(parts[1])) if len(parts) > 1 else 0
        logger.info(f'         think_tokens={think_len}, answer_tokens={answer_len}')

    del outputs
    torch.cuda.empty_cache()

peak_mem = torch.cuda.max_memory_allocated() / 1e9
logger.info(f'{"="*60}')
logger.info(f'PROBE RESULTS:')
logger.info(f'  Completions: {total} ({len(probe_problems)} problems x {G} generations)')
logger.info(f'  Lengths: mean={np.mean(all_lengths):.0f}, min={min(all_lengths)}, max={max(all_lengths)}')
logger.info(f'  Clipped ratio: {clipped}/{total} = {clipped/total:.1%}')
logger.info(f'  Peak GPU memory: {peak_mem:.1f} GB / {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
logger.info(f'  Headroom: {torch.cuda.get_device_properties(0).total_memory/1e9 - peak_mem:.1f} GB')
logger.info(f'  NOTE: Training backward pass uses ~1.5-2x more memory than generation!')
logger.info(f'  Estimated training peak: ~{peak_mem * 1.7:.0f} GB')
logger.info(f'{"="*60}')

if clipped/total > 0.8:
    logger.warning('WARNING: >80% clipped! Increase MAX_COMPLETION or disable thinking.')
elif clipped/total > 0.5:
    logger.warning('CAUTION: >50% clipped. Consider increasing MAX_COMPLETION.')
else:
    logger.success('OK: Clipped ratio acceptable. Proceed with training.')

FastLanguageModel.for_training(model)


12:58:27 | INFO     | Probe: 3 problems x G=8 completions, MAX_COMPLETION=4096
12:58:27 | INFO     | GPU before: 19.0 GB allocated, 19.0 GB peak
13:01:35 | INFO     |   [  easy] mean=4096, min=4096, max=4096, clipped=8/8, time=187.4s
13:01:35 | INFO     |          boxed=False, think=True, tokens=4096
13:01:35 | INFO     |          think_tokens=696, answer_tokens=196
13:03:06 | INFO     |   [medium] mean=2573, min=2573, max=2573, clipped=0/8, time=91.4s
13:03:06 | INFO     |          boxed=False, think=True, tokens=2573
13:03:06 | INFO     |          think_tokens=1158, answer_tokens=36
13:05:33 | INFO     |   [  hard] mean=4096, min=4096, max=4096, clipped=8/8, time=146.6s
13:05:33 | INFO     |          boxed=False, think=True, tokens=4096
13:05:33 | INFO     |          think_tokens=2047, answer_tokens=2048
13:05:33 | INFO     | ============================================================
13:05:33 | INFO     | PROBE RESULTS:
13:05:33 | INFO     |   Completions: 24 (3 problems x 8 genera

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3_5ForConditionalGeneration(
      (model): Qwen3_5Model(
        (visual): Qwen3_5VisionModel(
          (patch_embed): Qwen3_5VisionPatchEmbed(
            (proj): Conv3d(3, 1152, kernel_size=(2, 16, 16), stride=(2, 16, 16))
          )
          (pos_embed): Embedding(2304, 1152)
          (rotary_pos_emb): Qwen3_5VisionRotaryEmbedding()
          (blocks): ModuleList(
            (0-26): 27 x Qwen3_5VisionBlock(
              (norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
              (norm2): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
              (attn): Qwen3_5VisionAttention(
                (qkv): Linear(in_features=1152, out_features=3456, bias=True)
                (proj): Linear(in_features=1152, out_features=1152, bias=True)
              )
              (mlp): Qwen3_5VisionMLP(
                (linear_fc1): Linear(in_features=1152, out_features=4304, bias=True)
               

In [ ]:
# Verify bitsandbytes works for 8-bit optimizers
try:
    import bitsandbytes as bnb
    bnb.optim.AdamW8bit  # test native code is loadable
    _bnb_ok = True
except Exception:
    _bnb_ok = False
    logger.warning("bitsandbytes native code unavailable — falling back to adamw_torch_fused")
if not _bnb_ok and "8bit" in STAGE2_OPTIMIZER:
    STAGE2_OPTIMIZER = "adamw_torch_fused"
    logger.info(f"  Optimizer override: {STAGE2_OPTIMIZER}")

# ============================================================
# Training: two-stage curriculum with GDPO-style decoupled rewards
# Stage 1: easy+medium only (warm-up, AdamW, stabilize policy)
# Stage 2: all tiers, Lion optimizer, difficulty reweighting
# Optimizations: Dr. GRPO loss, ReDit dithering, Clip-Higher
# NOTE: NO zero-variance masking — uses ReDit dithering instead
# (with Colab disconnect recovery)
# ============================================================


# ============================================================
# GRPOLoggingCallback — detailed per-step metrics via loguru
# Solves: report_to="none" discards TRL metrics; no per-step visibility.
# ============================================================
from transformers import TrainerCallback
import time as _cb_time

class GRPOLoggingCallback(TrainerCallback):
    """Logs GRPO metrics (loss, reward, lengths, LR) at each logging step.

    Works with report_to="none" — intercepts on_log independently.
    Writes to both Colab console and persistent Drive log file.
    """

    def __init__(self):
        self._step_t0 = None
        self._train_t0 = None

    def on_train_begin(self, args, state, control, **kwargs):
        self._train_t0 = _cb_time.time()
        logger.info(
            f"Training started: {args.max_steps} steps, "
            f"batch={args.per_device_train_batch_size}x{args.gradient_accumulation_steps}, "
            f"G={args.num_generations}"
        )

    def on_step_begin(self, args, state, control, **kwargs):
        self._step_t0 = _cb_time.time()

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return
        step = state.global_step

        parts = [f'step={step}']

        if 'loss' in logs:
            parts.append(f'loss={logs["loss"]:.4f}')

        r = logs.get('reward', logs.get('reward/mean'))
        if r is not None:
            rs = logs.get('reward_std', logs.get('reward/std', 0))
            parts.append(f'reward={r:.3f}±{rs:.3f}')

        cl = logs.get('completion_length/mean')
        if cl is not None:
            parts.append(f'comp_len={cl:.0f}')

        lr = logs.get('learning_rate')
        if lr is not None:
            parts.append(f'lr={lr:.2e}')

        gn = logs.get('grad_norm')
        if gn is not None:
            parts.append(f'grad_norm={gn:.3f}')

        logger.info(" | ".join(parts))

        for k, v in sorted(logs.items()):
            if 'rewards/' in k:
                logger.debug(f'  {k}: {v:.4f}')

    def on_step_end(self, args, state, control, **kwargs):
        if self._step_t0 and state.global_step % (args.logging_steps * 10) == 0:
            elapsed = _cb_time.time() - self._step_t0
            remaining = elapsed * (args.max_steps - state.global_step)
            logger.debug(
                f"Step {state.global_step}/{args.max_steps} "
                f"({elapsed:.1f}s/step, ETA ~{remaining / 60:.0f}min)"
            )

    def on_train_end(self, args, state, control, **kwargs):
        total = _cb_time.time() - self._train_t0 if self._train_t0 else 0
        logger.success(
            f"Training finished: {state.global_step} steps in {total / 60:.1f}min"
        )


gspo_callback = GRPOLoggingCallback()


def find_latest_checkpoint(output_dir):
    """Find latest TRL checkpoint for resume after Colab disconnect.

    Validates that the checkpoint's LoRA rank matches the current LORA_R
    config to prevent size mismatch errors on resume.
    """
    if not os.path.isdir(output_dir):
        return None
    checkpoints = [d for d in os.listdir(output_dir) if d.startswith("checkpoint-")]
    if not checkpoints:
        return None
    # Check from newest to oldest — first compatible one wins
    for ckpt_name in sorted(checkpoints, key=lambda x: int(x.split("-")[1]), reverse=True):
        path = os.path.join(output_dir, ckpt_name)
        adapter_config_path = os.path.join(path, "adapter_config.json")
        if os.path.exists(adapter_config_path):
            with open(adapter_config_path, "r") as f:
                adapter_cfg = json.load(f)
            ckpt_r = adapter_cfg.get("r", None)
            if ckpt_r is not None and ckpt_r != LORA_R:
                logger.debug(f"  SKIP {ckpt_name}: LoRA r={ckpt_r} != current LORA_R={LORA_R}")
                continue
            ckpt_modules = set(adapter_cfg.get("target_modules", []))
            if ckpt_modules and ckpt_modules != set(TARGET_MODULES):
                logger.debug(f"  SKIP {ckpt_name}: target_modules mismatch (ckpt={sorted(ckpt_modules)}, current={sorted(TARGET_MODULES)})")
                continue
        logger.info(f"  Found compatible checkpoint: {path}")
        return path
    logger.info(f"  No compatible checkpoints found in {output_dir} (all have mismatched LoRA rank)")
    return None


def make_gspo_config(output_dir, max_steps, warmup_steps=15, **overrides):
    """Create GRPOConfig for a training stage.

    Unsloth wraps GRPOConfig with a fixed __init__ that blocks newer TRL
    params. We work around this by:
    1. Passing only params accepted by Unsloth's __init__
    2. Injecting the rest as attributes AFTER construction
    TRL's trainer reads config attrs at runtime, so injected params work.

    **overrides: stage-specific param overrides (e.g. optim, learning_rate
    for Lion in Stage 2). Applied on top of defaults.
    """
    import inspect

    # All desired params (defaults = Stage 1 / AdamW)
    all_kwargs = dict(
        output_dir=output_dir,
        max_steps=max_steps,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="cosine",
        warmup_steps=warmup_steps,
        num_generations=G,
        max_completion_length=MAX_COMPLETION,
        max_prompt_length=MAX_PROMPT_LENGTH,
        # Loss type: "sapo", "dr_grpo", "grpo", "dapo"
        loss_type=LOSS_TYPE,
        beta=BETA,
        # Clip-Higher (DAPO/VAPO)
        epsilon=EPSILON,
        epsilon_high=EPSILON_HIGH,
        # GSPO (arXiv 2507.18071)
        importance_sampling_level=IMPORTANCE_SAMPLING_LEVEL,
        steps_per_generation=STEPS_PER_GENERATION,
        mask_truncated_completions=False,  # False: truncated completions get 0 correctness (negative signal for being too long)
        # GDPO reward weights (arXiv 2601.05242)
        reward_weights=REWARD_WEIGHTS,
        # GDPO: normalize each reward independently, then weighted sum (arXiv 2601.05242)
        multi_objective_aggregation="normalize_then_sum",
        # Standard training params
        bf16=True,
        logging_steps=LOGGING_STEPS,
        save_steps=SAVE_STEPS,
        save_total_limit=3,
        optim="adamw_torch_fused",
        max_grad_norm=MAX_GRAD_NORM,
        # Optimizer stability (post-audit: DeepSeek-Math, VAPO best practices)
        weight_decay=WEIGHT_DECAY,
        adam_beta2=ADAM_BETA2,
        temperature=0.9,
        seed=42,
        report_to="none",
    )

    # Apply stage-specific overrides (e.g. Lion optimizer for Stage 2)
    if overrides:
        all_kwargs.update(overrides)
        logger.info(f"  Stage overrides applied: {list(overrides.keys())}")

    # Split: params accepted by __init__ vs post-init injection
    sig = inspect.signature(GRPOConfig.__init__)
    valid_init = set(sig.parameters.keys())

    init_kwargs = {}
    post_kwargs = {}
    for k, v in all_kwargs.items():
        if k in valid_init:
            init_kwargs[k] = v
        else:
            post_kwargs[k] = v

    config = GRPOConfig(**init_kwargs)

    # Inject remaining params as attributes
    for k, v in post_kwargs.items():
        setattr(config, k, v)

    if post_kwargs:
        logger.debug(f"  GRPOConfig: injected post-init: {list(post_kwargs.keys())}")

    return config


stage_metrics = {}
os.makedirs(OUTPUT_DIR, exist_ok=True)
stage1_dir = os.path.join(OUTPUT_DIR, "stage1")
stage2_dir = os.path.join(OUTPUT_DIR, "stage2")
stage1_adapter_path = os.path.join(OUTPUT_DIR, "stage1_adapter")

# Fix for TRL's GRPOTrainer trying to suppress warnings on a model that doesn't have the dictionary
if not hasattr(model, "warnings_issued"):
    model.warnings_issued = {}

# ========================
# Stage 1: Easy + Medium problems (warm-up, AdamW)
# ========================
stage1_steps = CURRICULUM_CONFIG["stage1_steps"]

# Check if stage1 adapter exists AND has matching LoRA rank
_stage1_done = False
_s1_adapter_file = os.path.join(stage1_adapter_path, "adapter_model.safetensors")
_s1_config_file = os.path.join(stage1_adapter_path, "adapter_config.json")
if os.path.exists(_s1_adapter_file):
    if os.path.exists(_s1_config_file):
        with open(_s1_config_file, "r") as f:
            _s1_cfg = json.load(f)
        if _s1_cfg.get("r") == LORA_R and set(_s1_cfg.get("target_modules", [])) == set(TARGET_MODULES):
            _stage1_done = True
        else:
            logger.info(f"  Stage 1 adapter mismatch (r={_s1_cfg.get('r')}, modules={len(_s1_cfg.get('target_modules', []))}) — retraining")
    else:
        # No config file — assume compatible (legacy checkpoint)
        _stage1_done = True

if _stage1_done:
    logger.info(f"\n{'='*60}")
    logger.debug(f"STAGE 1: SKIPPED (adapter found at {stage1_adapter_path})")
    logger.info(f"{'='*60}")
    from safetensors.torch import load_file as _load_s1
    model.load_state_dict(
        _load_s1(os.path.join(stage1_adapter_path, "adapter_model.safetensors")),
        strict=False,
    )
    logger.success("  Loaded stage 1 adapter weights")
else:
    logger.info(f"\n{'='*60}")
    logger.info(f"STAGE 1: Easy + Medium ({stage1_steps} steps) — AdamW")
    logger.info(f"{'='*60}")
    logger.info(f"  Problems: {len(easy_medium_problems)} (easy+medium only)")
    logger.info(f"  Optimizer: adamw_torch_fused, LR={LEARNING_RATE}")
    logger.info(f"  Loss: {LOSS_TYPE}")
    logger.info(f"  ReDit dithering: sigma={DITHERING_SIGMA} (no zero-variance masking)")
    logger.info(f"  GDPO reward_funcs: [correctness, format, socratic]")
    logger.info(f"  Reward weights: {REWARD_WEIGHTS} [correctness, format, socratic]")
    logger.info(f"  ThinkingBudgetProcessor: {THINKING_BUDGET} tokens")

    stage1_ds = format_problems_as_dataset(easy_medium_problems)
    stage1_config = make_gspo_config(
        output_dir=stage1_dir,
        max_steps=stage1_steps,
        warmup_steps=30,            # 10% of 300 steps
    )

    trainer_s1 = GRPOTrainer(
        model=model,
        args=stage1_config,
        train_dataset=stage1_ds,
        reward_funcs=[difficulty_weighted_correctness_fn, _base_format_fn, _base_socratic_fn],  # FIX: Socratic re-added (weight=0.45)
        processing_class=tokenizer,
        callbacks=[gspo_callback],
    )

    # Disable Unsloth gradient offloading (env var is ignored)
    if hasattr(model, '_offloaded_gradient_hooks'):
        for hook in model._offloaded_gradient_hooks:
            hook.remove()
        model._offloaded_gradient_hooks.clear()
        logger.debug("  Disabled Unsloth gradient offloading (keeping grads on GPU)")

    stage1_resume = find_latest_checkpoint(stage1_dir)
    logger.info("Starting Stage 1 training...")
    result_s1 = trainer_s1.train(resume_from_checkpoint=stage1_resume)

    stage_metrics["stage1"] = {
        "steps": stage1_steps,
        "final_loss": result_s1.training_loss,
        "metrics": result_s1.metrics,
        "problems": len(easy_medium_problems),
        "tiers": "easy+medium",
        "optimizer": "adamw_torch_fused",
    }
    logger.success(f"\nStage 1 complete! Loss: {result_s1.training_loss:.4f}")

    model.save_pretrained(stage1_adapter_path)
    tokenizer.save_pretrained(stage1_adapter_path)
    logger.success(f"  Stage 1 adapter saved to {stage1_adapter_path}")

    del trainer_s1
    torch.cuda.empty_cache()


# ========================
# Stage 2: All tiers with difficulty reweighting + Lion optimizer
# Lion (arXiv 2302.06675): sign-based momentum, robust to noisy RL gradients.
# Hyperparams adjusted per Lion paper: LR 3x smaller, weight_decay 3x larger.
# ========================
stage2_steps = CURRICULUM_CONFIG["stage2_steps"]
logger.info(f"\n{'='*60}")
logger.info(f"STAGE 2: All Tiers + Lion Optimizer ({stage2_steps} steps)")
logger.info(f"{'='*60}")
logger.info(f"  Problems: {len(verifiable_problems)} (all tiers)")
logger.info(f"  Difficulty weights: {DIFFICULTY_WEIGHTS}")
logger.info(f"  Optimizer: {STAGE2_OPTIMIZER} (sign-based momentum)")
logger.info(f"  LR={STAGE2_LEARNING_RATE}, wd={STAGE2_WEIGHT_DECAY}")
logger.info(f"  max_grad_norm={MAX_GRAD_NORM}")
logger.info(f"  GDPO reward_funcs: [correctness, format, socratic]")

stage2_ds = format_problems_as_dataset(verifiable_problems)
stage2_config = make_gspo_config(
    output_dir=stage2_dir,
    max_steps=stage2_steps,
    warmup_steps=40,                # 8% of 500 = 40 steps (longer warmup for hard problems)
    # Lion optimizer overrides
    optim=STAGE2_OPTIMIZER,
    learning_rate=STAGE2_LEARNING_RATE,
    weight_decay=STAGE2_WEIGHT_DECAY,
)

trainer_s2 = GRPOTrainer(
    model=model,
    args=stage2_config,
    train_dataset=stage2_ds,
    reward_funcs=[difficulty_weighted_correctness_fn, _base_format_fn, _base_socratic_fn],  # FIX: Socratic re-added (weight=0.45)
    processing_class=tokenizer,
    callbacks=[gspo_callback],
)

stage2_resume = find_latest_checkpoint(stage2_dir)
logger.info("Starting Stage 2 training...")
result_s2 = trainer_s2.train(resume_from_checkpoint=stage2_resume)

stage_metrics["stage2"] = {
    "steps": stage2_steps,
    "final_loss": result_s2.training_loss,
    "metrics": result_s2.metrics,
    "problems": len(verifiable_problems),
    "tiers": "all (easy+medium+hard)",
    "optimizer": STAGE2_OPTIMIZER,
    "learning_rate": STAGE2_LEARNING_RATE,
    "weight_decay": STAGE2_WEIGHT_DECAY,
    "difficulty_weights": DIFFICULTY_WEIGHTS,
}
logger.success(f"\nStage 2 complete! Loss: {result_s2.training_loss:.4f}")

trainer_s2.save_model(os.path.join(OUTPUT_DIR, "final"))
logger.success(f"\nCurriculum GSPO training complete!")
if "stage1" in stage_metrics:
    logger.info(f"  Stage 1 loss: {stage_metrics['stage1']['final_loss']:.4f} (AdamW)")
logger.info(f"  Stage 2 loss: {result_s2.training_loss:.4f} (Lion)")

training_log = {
    "total_steps": TOTAL_STEPS,
    "cold_start_sft_steps": COLD_START_SFT_STEPS,
    "thinking_budget": THINKING_BUDGET,
    "dithering_sigma": DITHERING_SIGMA,
    "lora_r": LORA_R,
    "stages": stage_metrics,
}


13:05:33 | INFO     | 
13:05:33 | INFO     | STAGE 1: Easy + Medium (300 steps) — AdamW
13:05:33 | INFO     | ============================================================
13:05:33 | INFO     |   Problems: 12740 (easy+medium only)
13:05:33 | INFO     |   Optimizer: adamw_torch_fused, LR=5e-07
13:05:33 | INFO     |   Loss: dr_grpo
13:05:33 | INFO     |   ReDit dithering: sigma=0.05 (no zero-variance masking)
13:05:33 | INFO     |   GDPO reward_funcs: [correctness, format, socratic]
13:05:33 | INFO     |   Reward weights: [0.4, 0.15, 0.45] [correctness, format, socratic]
13:05:33 | INFO     |   ThinkingBudgetProcessor: 2048 tokens
13:05:34 | DEBUG    |   GRPOConfig: injected post-init: ['multi_objective_aggregation']
13:05:34 | INFO     | Starting Stage 1 training...
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'e

Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
5,0.005737
10,-0.055703


13:18:51 | INFO     | step=5 | loss=0.0057 | reward=0.383±0.112 | lr=6.67e-08 | grad_norm=0.004
13:18:51 | DEBUG    |   rewards/_base_format_fn/mean: 0.7479
13:18:51 | DEBUG    |   rewards/_base_format_fn/std: 0.1942
13:18:51 | DEBUG    |   rewards/_base_socratic_fn/mean: 0.4625
13:18:51 | DEBUG    |   rewards/_base_socratic_fn/std: 0.2081
13:18:51 | DEBUG    |   rewards/difficulty_weighted_correctness_fn/mean: 0.1555
13:18:51 | DEBUG    |   rewards/difficulty_weighted_correctness_fn/std: 0.2495
13:29:58 | INFO     | step=10 | loss=-0.0557 | reward=0.390±0.169 | lr=1.50e-07 | grad_norm=0.004
13:29:58 | DEBUG    |   rewards/_base_format_fn/mean: 0.7500
13:29:58 | DEBUG    |   rewards/_base_format_fn/std: 0.2090
13:29:58 | DEBUG    |   rewards/_base_socratic_fn/mean: 0.5069
13:29:58 | DEBUG    |   rewards/_base_socratic_fn/std: 0.2934
13:29:58 | DEBUG    |   rewards/difficulty_weighted_correctness_fn/mean: 0.1246
13:29:58 | DEBUG    |   rewards/difficulty_weighted_correctness_fn/std: 0.2

In [ ]:
# ============================================================
# HOTFIX: Inject CSV metrics logger into running callback
# Run this cell ONCE while training is in progress.
# Collects structured per-step data for diploma plots/tables.
# ============================================================
import csv
import os
import time as _hf_time
from pathlib import Path

# ---- Metrics CSV on Drive (survives Colab disconnect) ----
METRICS_DIR = os.path.join(OUTPUT_DIR, "metrics")
os.makedirs(METRICS_DIR, exist_ok=True)
METRICS_CSV = os.path.join(METRICS_DIR, "training_metrics.csv")

CSV_COLUMNS = [
    "timestamp", "stage", "step",
    "loss", "reward_mean", "reward_std",
    "correctness_mean", "correctness_std",
    "format_mean", "format_std",
    "socratic_mean", "socratic_std",
    "learning_rate", "grad_norm",
    "completion_length_mean",
    "epoch",
    "prompt_mode",  # socratic/taskgen ratio in this batch
]

# Create CSV with header if it doesn't exist
if not os.path.exists(METRICS_CSV):
    with open(METRICS_CSV, "w", newline="") as f:
        csv.DictWriter(f, fieldnames=CSV_COLUMNS).writeheader()
    logger.info(f"Created metrics CSV: {METRICS_CSV}")
else:
    logger.info(f"Appending to existing metrics CSV: {METRICS_CSV}")


def _on_log_with_csv(self, args, state, control, logs=None, **kwargs):
    """Enhanced on_log: loguru console + CSV file for structured analysis."""
    if not logs:
        return
    step = state.global_step

    # ---- Console logging (same as before) ----
    parts = [f'step={step}']
    if 'loss' in logs:
        parts.append(f'loss={logs["loss"]:.4f}')
    r = logs.get('reward', logs.get('reward/mean'))
    if r is not None:
        rs = logs.get('reward_std', logs.get('reward/std', 0))
        parts.append(f'reward={r:.3f}\u00b1{rs:.3f}')
    cl = logs.get('completion_length/mean')
    if cl is not None:
        parts.append(f'comp_len={cl:.0f}')
    lr = logs.get('learning_rate')
    if lr is not None:
        parts.append(f'lr={lr:.2e}')
    gn = logs.get('grad_norm')
    if gn is not None:
        parts.append(f'grad_norm={gn:.3f}')
    logger.info(" | ".join(parts))

    for k, v in sorted(logs.items()):
        if 'rewards/' in k:
            logger.debug(f'  {k}: {v:.4f}')

    # ---- CSV structured logging ----
    # Determine current stage from step number
    _stage1_steps = CURRICULUM_CONFIG.get("stage1_steps", 300)
    current_stage = "stage1" if step <= _stage1_steps else "stage2"

    row = {
        "timestamp": _hf_time.strftime("%Y-%m-%d %H:%M:%S"),
        "stage": current_stage,
        "step": step,
        "loss": logs.get("loss"),
        "reward_mean": logs.get("reward", logs.get("reward/mean")),
        "reward_std": logs.get("reward_std", logs.get("reward/std")),
        "correctness_mean": logs.get("rewards/difficulty_weighted_correctness_fn/mean"),
        "correctness_std": logs.get("rewards/difficulty_weighted_correctness_fn/std"),
        "format_mean": logs.get("rewards/format_fn/mean",
                        logs.get("rewards/_base_format_fn/mean")),
        "format_std": logs.get("rewards/format_fn/std",
                       logs.get("rewards/_base_format_fn/std")),
        "socratic_mean": logs.get("rewards/_base_socratic_fn/mean"),
        "socratic_std": logs.get("rewards/_base_socratic_fn/std"),
        "learning_rate": logs.get("learning_rate"),
        "grad_norm": logs.get("grad_norm"),
        "completion_length_mean": logs.get("completion_length/mean"),
        "epoch": logs.get("epoch"),
    }

    try:
        with open(METRICS_CSV, "a", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=CSV_COLUMNS)
            writer.writerow(row)
    except Exception as e:
        logger.warning(f"Failed to write metrics CSV: {e}")


# Apply monkey-patch to the existing callback class
GRPOLoggingCallback.on_log = _on_log_with_csv
logger.success(f"CSV metrics logger patched — writing to {METRICS_CSV}")
logger.info("Columns: step, loss, reward, correctness, format, socratic, lr, grad_norm, completion_length")


11:03:38 | INFO     | Created metrics CSV: /content/drive/MyDrive/MITS/checkpoints/gspo_qwen3.5_9b_v2/metrics/training_metrics.csv
11:03:38 | SUCCESS  | CSV metrics logger patched — writing to /content/drive/MyDrive/MITS/checkpoints/gspo_qwen3.5_9b_v2/metrics/training_metrics.csv
11:03:38 | INFO     | Columns: step, loss, reward, correctness, format, socratic, lr, grad_norm, completion_length


In [ ]:
# ============================================================
# OPTIONAL: Recover past metrics from loguru logs
# Parses the loguru text output to backfill CSV for steps
# that occurred before the CSV logger was patched.
# Run ONCE after patching the CSV logger above.
# ============================================================
import re
import csv
import glob
import pandas as pd
# Find loguru log files on Drive
log_patterns = [
    os.path.join(OUTPUT_DIR, "*.log"),
    os.path.join(OUTPUT_DIR, "metrics", "*.log"),
    "/content/drive/MyDrive/MITS/gspo_training.log",
]

log_lines = []
for pattern in log_patterns:
    for f in glob.glob(pattern):
        with open(f, "r") as fh:
            log_lines.extend(fh.readlines())
        logger.info(f"  Read {f}: {len(log_lines)} lines")

# Also try to parse from notebook output (trainer_state.json)
trainer_state_paths = [
    os.path.join(OUTPUT_DIR, "stage1", "checkpoint-*", "trainer_state.json"),
]
for pattern in trainer_state_paths:
    for f in glob.glob(pattern):
        import json as _json_recover
        with open(f, "r") as fh:
            state = _json_recover.load(fh)
        if "log_history" in state:
            logger.info(f"  Found trainer_state.json with {len(state['log_history'])} log entries")
            # trainer_state.json has the most reliable metrics
            existing_steps = set()
            if os.path.exists(METRICS_CSV):
                edf = pd.read_csv(METRICS_CSV)
                existing_steps = set(edf["step"].astype(int).tolist())

            recovered = 0
            for entry in state["log_history"]:
                step = entry.get("step", 0)
                if step in existing_steps or step == 0:
                    continue
                if "loss" not in entry:
                    continue  # skip eval/other entries

                _s1_steps = CURRICULUM_CONFIG.get("stage1_steps", 300)
                row = {
                    "timestamp": "",
                    "stage": "stage1" if step <= _s1_steps else "stage2",
                    "step": step,
                    "loss": entry.get("loss"),
                    "reward_mean": entry.get("reward", entry.get("reward/mean")),
                    "reward_std": entry.get("reward_std", entry.get("reward/std")),
                    "correctness_mean": entry.get("rewards/difficulty_weighted_correctness_fn/mean"),
                    "correctness_std": entry.get("rewards/difficulty_weighted_correctness_fn/std"),
                    "format_mean": entry.get("rewards/format_fn/mean"),
                    "format_std": entry.get("rewards/format_fn/std"),
                    "learning_rate": entry.get("learning_rate"),
                    "grad_norm": entry.get("grad_norm"),
                    "completion_length_mean": entry.get("completion_length/mean"),
                    "epoch": entry.get("epoch"),
                }
                with open(METRICS_CSV, "a", newline="") as fh:
                    csv.DictWriter(fh, fieldnames=CSV_COLUMNS).writerow(row)
                recovered += 1
                existing_steps.add(step)

            logger.success(f"  Recovered {recovered} steps from trainer_state.json")

# Sort CSV by step for clean plotting
if os.path.exists(METRICS_CSV):
    df_sort = pd.read_csv(METRICS_CSV)
    df_sort = df_sort.sort_values("step").drop_duplicates(subset=["step"], keep="last")
    df_sort.to_csv(METRICS_CSV, index=False)
    logger.success(f"Metrics CSV: {len(df_sort)} rows, steps {df_sort['step'].min()}-{df_sort['step'].max()}")


22:58:41 | INFO     |   Read /content/drive/MyDrive/MITS/checkpoints/gspo_qwen3.5_9b/gspo_training.log: 1383 lines
22:58:41 | INFO     |   Found trainer_state.json with 10 log entries
22:58:41 | SUCCESS  |   Recovered 10 steps from trainer_state.json
22:58:41 | INFO     |   Found trainer_state.json with 20 log entries
22:58:41 | SUCCESS  |   Recovered 10 steps from trainer_state.json
22:58:41 | SUCCESS  | Metrics CSV: 20 rows, steps 5-100


In [ ]:
from huggingface_hub import login

# Login interactively (token NOT hardcoded for security)
login()


In [ ]:
# ============================================================
# Save final adapter + training config
# ============================================================

final_adapter_path = os.path.join(OUTPUT_DIR, "final_adapter")
model.save_pretrained(final_adapter_path)
tokenizer.save_pretrained(final_adapter_path)
logger.success(f"Final GSPO adapter saved to {final_adapter_path}")

config_to_save = {
    "stage": "gspo",
    "pipeline": "GSPO (triple reward) -> KTO -> DPO",
    "pipeline_position": "1 of 3",
    "base_model": BASE_MODEL,
    "note": "No SFT stage — Instruct model used directly",
    "hardware": GPU_TYPE,
    "importance_sampling_level": IMPORTANCE_SAMPLING_LEVEL,
    "loss_type": LOSS_TYPE,
    "beta": BETA,
    "epsilon": EPSILON,
    "epsilon_high": EPSILON_HIGH,
    "max_grad_norm": MAX_GRAD_NORM,
    "max_seq_length": MAX_SEQ_LENGTH,
    "max_prompt_length": MAX_PROMPT_LENGTH,
    "max_completion_length": MAX_COMPLETION,
    # Stage 1: AdamW
    "stage1_optimizer": "adamw_torch_fused",
    "stage1_learning_rate": LEARNING_RATE,
    "stage1_weight_decay": WEIGHT_DECAY,
    "stage1_adam_beta2": ADAM_BETA2,
    # Stage 2: Lion (arXiv 2302.06675)
    "stage2_optimizer": STAGE2_OPTIMIZER,
    "stage2_learning_rate": STAGE2_LEARNING_RATE,
    "stage2_weight_decay": STAGE2_WEIGHT_DECAY,
    "stage2_note": "Lion sign-based momentum — robust to noisy RL gradients",
    # Common
    "steps_per_generation": STEPS_PER_GENERATION,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "mask_truncated_completions": False,
    "G": G,
    "max_completion": MAX_COMPLETION,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "total_steps": TOTAL_STEPS,
    "total_problems": len(verifiable_problems),
    # ReDit (arXiv 2506.18631)
    "dithering_sigma": DITHERING_SIGMA,
    # Curriculum config
    "curriculum_config": CURRICULUM_CONFIG,
    "difficulty_distribution": dict(difficulty_dist),
    # GDPO
    "reward_approach": "GDPO decoupled (arXiv 2601.05242)",
    "reward_weights": REWARD_WEIGHTS,
    "reward_weights_names": ["correctness", "format", "socratic"],
    "reward_scale": "correct=1.0, wrong=0.0 (no negative penalties)",
    # Post-mortem fixes (2026-03-23)
    "postmortem_fixes": {
        "system_prompt": "Socratic (matches deployment) — Llama 2 Ghost Attention",
        "beta": "0.0 -> 0.04 (KL penalty for instruction-following)",
        "reward_weights": "[0.85,0.15] -> [0.4,0.15,0.45] (Socratic re-added)",
        "cold_start_socratic": f"{COLD_START_SOCRATIC_STEPS} steps on dialogs.jsonl",
        "multi_mode_prompt": "78% Socratic, 22% TaskGen (prevents alignment tax)",
        "lora_rank": "kept r=16 (Tina: r=64 UNDERPERFORMS r=16)",
    },
    "cold_start_math_steps": COLD_START_SFT_STEPS,
    "cold_start_socratic_steps": COLD_START_SOCRATIC_STEPS,
    "prompt_mode_weights": {"socratic": 0.78, "taskgen": 0.22},
    "difficulty_weighting": "GRPO-LEAD (easy=0.4, medium=1.0, hard=1.5)",
    "zero_variance_masking": False,
    # Stage metrics
    "stage_metrics": stage_metrics,
    "references": [
        "GSPO arXiv:2507.18071",
        "Lion arXiv:2302.06675",
        "ReDit arXiv:2506.18631",
        "Dr. GRPO arXiv:2503.20783",
        "DAPO arXiv:2503.14476",
        "VAPO arXiv:2504.05118",
        "GDPO arXiv:2601.05242",
        "DRPO arXiv:2510.04474",
        "GRPO-LEAD arXiv:2504.09696",
        "Revisiting GRPO arXiv:2505.22257",
        "Llama 2 Ghost Attention arXiv:2307.09288",
        "MO-GRPO arXiv:2509.22047",
        "Tina arXiv:2504.15777",
        "LLD Death Spiral arXiv:2512.04220",
        "LoRA Safety arXiv:2507.17075",
    ],
}
config_path = os.path.join(OUTPUT_DIR, "training_config.json")
with open(config_path, "w") as f:
    json.dump(config_to_save, f, indent=2)
logger.success(f"Config saved to {config_path}")

# Optional: push to HF
PUSH_TO_HUB = True
HF_REPO_ID = "Siesher/mits-qwen3-9b-gspo"

if PUSH_TO_HUB:
    model.push_to_hub(HF_REPO_ID, private=True)
    tokenizer.push_to_hub(HF_REPO_ID, private=True)
    logger.success(f"Pushed to https://huggingface.co/{HF_REPO_ID}")

logger.success("\nDone! GSPO adapter ready for KTO (next stage).")

13:35:33 | SUCCESS  | Final GSPO adapter saved to /content/drive/MyDrive/MITS/checkpoints/gspo_qwen3.5_9b/final_adapter
13:35:33 | SUCCESS  | Config saved to /content/drive/MyDrive/MITS/checkpoints/gspo_qwen3.5_9b/training_config.json


README.md:   0%|          | 0.00/523 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 52.9kB /  173MB            

Saved model to https://huggingface.co/Siesher/mits-qwen3-9b-gspo


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp6mwtci1h/tokenizer.json: 100%|##########| 20.0MB / 20.0MB            

13:35:43 | SUCCESS  | Pushed to https://huggingface.co/Siesher/mits-qwen3-9b-gspo
13:35:43 | SUCCESS  | 
Done! GSPO adapter ready for KTO (next stage).


In [ ]:
# ============================================================
# Diploma/Paper: Training Metrics Visualization
# Run AFTER training completes (or mid-training for partial plots)
# Reads from: {OUTPUT_DIR}/metrics/training_metrics.csv
# Outputs: plots saved to {OUTPUT_DIR}/metrics/figures/
# ============================================================
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
from pathlib import Path

METRICS_CSV = os.path.join(OUTPUT_DIR, "metrics", "training_metrics.csv")
FIGURES_DIR = os.path.join(OUTPUT_DIR, "metrics", "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)

# ---- Load metrics ----
df = pd.read_csv(METRICS_CSV)
df["step"] = df["step"].astype(int)
logger.info(f"Loaded {len(df)} metric rows from {METRICS_CSV}")
logger.info(f"Steps: {df['step'].min()} → {df['step'].max()}, Stages: {df['stage'].unique().tolist()}")

# ---- Style ----
plt.rcParams.update({
    "figure.figsize": (12, 4),
    "font.size": 11,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
})
STAGE_COLORS = {"stage1": "#2196F3", "stage2": "#FF9800"}

def save_fig(fig, name):
    path = os.path.join(FIGURES_DIR, f"{name}.png")
    fig.savefig(path, dpi=200, bbox_inches="tight")
    fig.savefig(path.replace(".png", ".pdf"), bbox_inches="tight")  # vector for paper
    logger.info(f"  Saved: {path}")
    plt.close(fig)


# ============================================================
# Figure 1: Training Loss + Reward (dual axis)
# ============================================================
fig, ax1 = plt.subplots()
ax2 = ax1.twinx()

for stage, grp in df.groupby("stage"):
    c = STAGE_COLORS.get(stage, "gray")
    ax1.plot(grp["step"], grp["loss"], color=c, alpha=0.7, label=f"Loss ({stage})")
    ax2.plot(grp["step"], grp["reward_mean"], color=c, linestyle="--", alpha=0.7, label=f"Reward ({stage})")
    if grp["reward_std"].notna().any():
        ax2.fill_between(grp["step"],
                         grp["reward_mean"] - grp["reward_std"],
                         grp["reward_mean"] + grp["reward_std"],
                         alpha=0.1, color=c)

ax1.set_xlabel("Training Step")
ax1.set_ylabel("GRPO Loss", color="#2196F3")
ax2.set_ylabel("Mean Reward", color="#FF9800")
ax1.set_title("GSPO Training: Loss and Reward Dynamics")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right", fontsize=9)
save_fig(fig, "01_loss_reward")


# ============================================================
# Figure 2: Reward Decomposition (correctness vs format)
# ============================================================
fig, ax = plt.subplots()
for stage, grp in df.groupby("stage"):
    c = STAGE_COLORS.get(stage, "gray")
    if grp["correctness_mean"].notna().any():
        ax.plot(grp["step"], grp["correctness_mean"], color=c, label=f"Correctness ({stage})")
        ax.fill_between(grp["step"],
                        grp["correctness_mean"] - grp["correctness_std"].fillna(0),
                        grp["correctness_mean"] + grp["correctness_std"].fillna(0),
                        alpha=0.1, color=c)
    if grp["format_mean"].notna().any():
        ax.plot(grp["step"], grp["format_mean"], color=c, linestyle=":", label=f"Format ({stage})")

ax.set_xlabel("Training Step")
ax.set_ylabel("Reward Value")
ax.set_title("GSPO: Reward Decomposition (GDPO Decoupled)")
ax.set_ylim(0, 1.05)
ax.legend(fontsize=9)
save_fig(fig, "02_reward_decomposition")


# ============================================================
# Figure 3: Learning Rate Schedule
# ============================================================
fig, ax = plt.subplots(figsize=(10, 3))
for stage, grp in df.groupby("stage"):
    c = STAGE_COLORS.get(stage, "gray")
    if grp["learning_rate"].notna().any():
        ax.plot(grp["step"], grp["learning_rate"], color=c, label=stage)
ax.set_xlabel("Training Step")
ax.set_ylabel("Learning Rate")
ax.set_title("Cosine LR Schedule (Stage 1: AdamW, Stage 2: Lion)")
ax.yaxis.set_major_formatter(mticker.ScalarFormatter(useMathText=True))
ax.ticklabel_format(style="sci", axis="y", scilimits=(0, 0))
ax.legend(fontsize=9)
save_fig(fig, "03_learning_rate")


# ============================================================
# Figure 4: Gradient Norm (training stability indicator)
# ============================================================
fig, ax = plt.subplots(figsize=(10, 3))
for stage, grp in df.groupby("stage"):
    c = STAGE_COLORS.get(stage, "gray")
    if grp["grad_norm"].notna().any():
        ax.plot(grp["step"], grp["grad_norm"], color=c, alpha=0.7, label=stage)
ax.set_xlabel("Training Step")
ax.set_ylabel("Gradient Norm")
ax.set_title("Gradient Norm During Training")
ax.legend(fontsize=9)
save_fig(fig, "04_gradient_norm")


# ============================================================
# ============================================================
# Figure 5: Socratic Reward (key training signal for tutoring style)
# ============================================================
fig, ax = plt.subplots()
for stage, grp in df.groupby("stage"):
    c = STAGE_COLORS.get(stage, "gray")
    if "socratic_mean" in grp.columns and grp["socratic_mean"].notna().any():
        ax.plot(grp["step"], grp["socratic_mean"], color=c, label=f"Socratic ({stage})")
        ax.fill_between(grp["step"],
                        grp["socratic_mean"] - grp["socratic_std"].fillna(0),
                        grp["socratic_mean"] + grp["socratic_std"].fillna(0),
                        alpha=0.1, color=c)
    if "correctness_mean" in grp.columns and grp["correctness_mean"].notna().any():
        ax.plot(grp["step"], grp["correctness_mean"], color=c, linestyle="--",
                alpha=0.5, label=f"Correctness ({stage})")

ax.set_xlabel("Training Step")
ax.set_ylabel("Reward Value")
ax.set_title("GSPO: Socratic vs Correctness Reward Dynamics")
ax.set_ylim(0, 1.05)
ax.legend(fontsize=9)
save_fig(fig, "05_socratic_reward")


# ============================================================
# Figure 6: Reward Decomposition — All 3 Components
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
reward_cols = [
    ("correctness_mean", "correctness_std", "Correctness (w=0.4)", "#2196F3"),
    ("format_mean", "format_std", "Format (w=0.15)", "#4CAF50"),
    ("socratic_mean", "socratic_std", "Socratic (w=0.45)", "#FF5722"),
]

for ax, (mean_col, std_col, title, color) in zip(axes, reward_cols):
    for stage, grp in df.groupby("stage"):
        if mean_col in grp.columns and grp[mean_col].notna().any():
            ax.plot(grp["step"], grp[mean_col], color=color, alpha=0.8,
                    label=stage, linestyle="-" if stage == "stage1" else "--")
            if std_col in grp.columns:
                ax.fill_between(grp["step"],
                                grp[mean_col] - grp[std_col].fillna(0),
                                grp[mean_col] + grp[std_col].fillna(0),
                                alpha=0.1, color=color)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("Step")
    ax.legend(fontsize=8)
    ax.set_ylim(0, 1.05)

axes[0].set_ylabel("Reward Value")
fig.suptitle("GSPO Triple GDPO Reward Decomposition", fontsize=12)
fig.tight_layout()
save_fig(fig, "06_triple_reward_decomposition")


# Summary Table (for paper)
# ============================================================
summary_rows = []
for stage, grp in df.groupby("stage"):
    summary_rows.append({
        "Stage": stage,
        "Steps": f"{grp['step'].min()}–{grp['step'].max()}",
        "Final Loss": f"{grp['loss'].iloc[-1]:.4f}" if grp['loss'].notna().any() else "—",
        "Avg Correctness": f"{grp['correctness_mean'].mean():.3f}" if grp['correctness_mean'].notna().any() else "—",
        "Avg Format": f"{grp['format_mean'].mean():.3f}" if grp['format_mean'].notna().any() else "—",
        "Avg Socratic": f"{grp['socratic_mean'].mean():.3f}" if 'socratic_mean' in grp.columns and grp['socratic_mean'].notna().any() else "—",
        "Avg Reward": f"{grp['reward_mean'].mean():.3f}" if grp['reward_mean'].notna().any() else "—",
        "Max Grad Norm": f"{grp['grad_norm'].max():.4f}" if grp['grad_norm'].notna().any() else "—",
    })

summary_df = pd.DataFrame(summary_rows)
print("\n" + "=" * 80)
print("GSPO Training Summary (for paper/diploma)")
print("=" * 80)
print(summary_df.to_string(index=False))
print("=" * 80)

# Save summary as LaTeX table
latex_path = os.path.join(FIGURES_DIR, "training_summary.tex")
summary_df.to_latex(latex_path, index=False, caption="GSPO training results per stage", label="tab:gspo_results")
logger.success(f"Summary LaTeX table: {latex_path}")
logger.success(f"All figures saved to: {FIGURES_DIR}")


13:40:13 | INFO     | Loaded 160 metric rows from /content/drive/MyDrive/MITS/checkpoints/gspo_qwen3.5_9b/metrics/training_metrics.csv
13:40:13 | INFO     | Steps: 5 → 500, Stages: ['stage1', 'stage2']
13:40:13 | INFO     |   Saved: /content/drive/MyDrive/MITS/checkpoints/gspo_qwen3.5_9b/metrics/figures/01_loss_reward.png
13:40:14 | INFO     |   Saved: /content/drive/MyDrive/MITS/checkpoints/gspo_qwen3.5_9b/metrics/figures/02_reward_decomposition.png
13:40:14 | INFO     |   Saved: /content/drive/MyDrive/MITS/checkpoints/gspo_qwen3.5_9b/metrics/figures/03_learning_rate.png
13:40:14 | INFO     |   Saved: /content/drive/MyDrive/MITS/checkpoints/gspo_qwen3.5_9b/metrics/figures/04_gradient_norm.png
13:40:14 | SUCCESS  | Summary LaTeX table: /content/drive/MyDrive/MITS/checkpoints/gspo_qwen3.5_9b/metrics/figures/training_summary.tex
13:40:14 | SUCCESS  | All figures saved to: /content/drive/MyDrive/MITS/checkpoints/gspo_qwen3.5_9b/metrics/figures



GSPO Training Summary (for paper/diploma)
 Stage Steps Final Loss Avg Correctness Avg Format Avg Reward Max Grad Norm
stage1 5–300     0.0474           0.547      0.906      0.575        0.0088
stage2 5–500     0.0288           0.547      0.849      0.593        0.0088


In [ ]:
# ============================================================
# Evaluation removed — run locally:
#   python training/scripts/evaluate_stage.py --stage gspo \
#     --adapter /content/drive/MyDrive/MITS/checkpoints/gspo_qwen3.5_9b/final
# ============================================================
logger.debug("Post-training evaluation skipped (run locally)")

In [ ]:
import os
import json
import glob
import pandas as pd

print("Ищем сохраненные логи обучения во всех чекпоинтах...")
metrics_list = []

# Ищем все файлы trainer_state.json в папках stage1 и stage2
state_files = glob.glob(os.path.join(OUTPUT_DIR, "stage*", "checkpoint-*", "trainer_state.json"))
print(f"Найдено файлов с логами: {len(state_files)}")

for f in state_files:
    # Определяем стадию по пути к файлу
    stage_name = "stage1" if "stage1" in f else "stage2"
    with open(f, "r") as fh:
        state = json.load(fh)
        if "log_history" in state:
            for entry in state["log_history"]:
                step = entry.get("step", 0)
                if step == 0 or "loss" not in entry:
                    continue

                row = {
                    "timestamp": "",
                    "stage": stage_name,
                    "step": step,
                    "loss": entry.get("loss"),
                    "reward_mean": entry.get("reward", entry.get("reward/mean")),
                    "reward_std": entry.get("reward_std", entry.get("reward/std")),
                    "correctness_mean": entry.get("rewards/difficulty_weighted_correctness_fn/mean"),
                    "correctness_std": entry.get("rewards/difficulty_weighted_correctness_fn/std"),
                    "format_mean": entry.get("rewards/format_fn/mean"),
                    "format_std": entry.get("rewards/format_fn/std"),
                    "learning_rate": entry.get("learning_rate"),
                    "grad_norm": entry.get("grad_norm"),
                    "completion_length_mean": entry.get("completion_length/mean"),
                    "epoch": entry.get("epoch"),
                }
                metrics_list.append(row)

df_recovered = pd.DataFrame(metrics_list)

if not df_recovered.empty:
    # Убираем дубликаты и сортируем
    df_recovered = df_recovered.sort_values(["stage", "step"]).drop_duplicates(subset=["stage", "step"], keep="last")

    # Сохраняем обратно в CSV
    csv_path = os.path.join(OUTPUT_DIR, "metrics", "training_metrics.csv")
    df_recovered.to_csv(csv_path, index=False)
    print(f"\n✅ Успешно восстановлено {len(df_recovered)} записей метрик для всех стадий!")

    print("\n=== СВОДКА ПО СТАДИЯМ ИЗ ВОССТАНОВЛЕННЫХ ЛОГОВ ===")
    for stage, grp in df_recovered.groupby("stage"):
        print(f"\n[{stage.upper()}]")
        print(f"  Записей лога: {len(grp)}")
        print(f"  Шаги: {grp['step'].min()} -> {grp['step'].max()}")
        print(f"  Начальный loss: {grp.iloc[0]['loss']:.4f}")
        print(f"  Финальный loss: {grp.iloc[-1]['loss']:.4f}")
        if pd.notna(grp.iloc[-1]['reward_mean']):
            print(f"  Финальная награда: {grp.iloc[-1]['reward_mean']:.4f}")

    print("\nПоследние записи для каждой стадии:")
    display(df_recovered.groupby("stage").tail(2)[['stage', 'step', 'loss', 'reward_mean', 'correctness_mean', 'learning_rate']])
else:
    print("❌ Не удалось найти детальные логи. Возможно, чекпоинты (папки checkpoint-*) были удалены.")


Ищем сохраненные логи обучения во всех чекпоинтах...
Найдено файлов с логами: 6

✅ Успешно восстановлено 160 записей метрик для всех стадий!

=== СВОДКА ПО СТАДИЯМ ИЗ ВОССТАНОВЛЕННЫХ ЛОГОВ ===

[STAGE1]
  Записей лога: 60
  Шаги: 5 -> 300
  Начальный loss: 0.0188
  Финальный loss: 0.0474
  Финальная награда: 0.6022

[STAGE2]
  Записей лога: 100
  Шаги: 5 -> 500
  Начальный loss: -0.0042
  Финальный loss: 0.0288
  Финальная награда: 0.5581

Последние записи для каждой стадии:


,stage,step,loss,reward_mean,correctness_mean,learning_rate
118,stage1,295,0.038794,0.481231,0.413728,6.089874e-10
119,stage1,300,0.047394,0.602208,0.553186,1.692300e-11
408,stage2,495,0.025916,0.473718,0.415256,8.394519e-11
409,stage2,500,0.028799,0.558129,0.507064,2.332128e-12


In [ ]:
import json
import os
from pprint import pprint
import pandas as pd

config_path = os.path.join(OUTPUT_DIR, "training_config.json")
csv_path = os.path.join(OUTPUT_DIR, "metrics", "training_metrics.csv")

print("=" * 60)
print("ПРОВЕРКА СТАТУСА ОБУЧЕНИЯ")
print("=" * 60)

# 1. Проверяем наличие финальной модели
final_adapter_dir = os.path.join(OUTPUT_DIR, "final_adapter")
if os.path.exists(final_adapter_dir):
    print("✅ Финальный адаптер успешно сохранен:", final_adapter_dir)
else:
    print("❌ Финальный адаптер НЕ НАЙДЕН. Возможно, обучение прервалось.")

# 2. Читаем итоговые метрики из JSON
if os.path.exists(config_path):
    with open(config_path, "r") as f:
        config_data = json.load(f)
    print("\n✅ Файл конфигурации и метрик найден.")
    print("\n--- ИТОГОВЫЕ МЕТРИКИ ПО СТАДИЯМ ---")
    if "stage_metrics" in config_data:
        for stage, metrics in config_data["stage_metrics"].items():
            print(f"\n[{stage.upper()}]")
            print(f"  Шагов пройдено: {metrics.get('steps', 'N/A')}")
            print(f"  Финальный loss: {metrics.get('final_loss', 'N/A')}")
            print(f"  Оптимизатор: {metrics.get('optimizer', 'N/A')}")
            if 'metrics' in metrics:
                print(f"  Время обучения: {metrics['metrics'].get('train_runtime', 'N/A')} сек.")
                print(f"  Шагов в секунду: {metrics['metrics'].get('train_steps_per_second', 'N/A')}")
else:
    print("\n❌ Файл training_config.json не найден.")

# 3. Смотрим последнюю статистику из CSV
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    if not df.empty:
        print("\n✅ CSV лог с метриками найден (всего записей:", len(df), ")")
        print("Последние 3 записи метрик:")
        display(df[['stage', 'step', 'loss', 'reward_mean', 'correctness_mean', 'format_mean']].tail(3))
else:
    print("\n❌ CSV файл с метриками не найден.")


ПРОВЕРКА СТАТУСА ОБУЧЕНИЯ
✅ Финальный адаптер успешно сохранен: /content/drive/MyDrive/MITS/checkpoints/gspo_qwen3.5_9b/final_adapter

✅ Файл конфигурации и метрик найден.

--- ИТОГОВЫЕ МЕТРИКИ ПО СТАДИЯМ ---

[STAGE2]
  Шагов пройдено: 500
  Финальный loss: 5.336627066730263e-06
  Оптимизатор: lion_8bit
  Время обучения: 58.0485 сек.
  Шагов в секунду: 8.613

✅ CSV лог с метриками найден (всего записей: 20 )
Последние 3 записи метрик:


,stage,step,loss,reward_mean,correctness_mean,format_mean
17,stage1,90,0.031359,0.605884,0.671085,0.87875
18,stage1,95,-0.011230,0.511400,0.515214,0.98750
19,stage1,100,0.058837,0.496027,0.505038,0.93000


In [ ]:
# ============================================================
# Export GGUF directly to Hugging Face (Bypasses local disk limits)
# ============================================================
# Unsloth merges LoRA weights and quantizes to GGUF, streaming
# directly to HF to avoid running out of Google Drive/Colab space.

HF_REPO_GGUF = "Siesher/mits-tutor-qwen3.5-9b-gguf"
QUANTIZATION = "q8_0"  # Options: q4_k_m, q5_k_m, q8_0

logger.info(f"Pushing GGUF ({QUANTIZATION}) directly to Hugging Face...")
logger.info(f"  Target repo: {HF_REPO_GGUF}")

# push_to_hub_gguf streams to HF directly
model.push_to_hub_gguf(
    HF_REPO_GGUF,
    tokenizer,
    quantization_method=QUANTIZATION,
)

logger.success(f"\n✅ GGUF exported and pushed to Hugging Face!")
logger.info(f"   Link: https://huggingface.co/{HF_REPO_GGUF}")
logger.info(f"\n📋 Local deployment steps:")
logger.info(f"   1. Go to https://huggingface.co/{HF_REPO_GGUF}/tree/main on your PC")
logger.info(f"   2. Download the .gguf file")
logger.info(f"   3. Rename to mits-tutor-qwen3.5-9b-q8_0.gguf")
logger.info(f"   4. Place next to training/Modelfile")
logger.info(f"   5. Run: ollama create mits-tutor -f training/Modelfile")
logger.info(f"   6. Run: ollama run mits-tutor")


13:53:26 | INFO     | Pushing GGUF (q8_0) directly to Hugging Face...
13:53:26 | INFO     |   Target repo: Siesher/mits-tutor-qwen3.5-9b-gguf


Unsloth: Converting model to GGUF format...
Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...



Unsloth: Copying 4 files from cache to `/tmp/unsloth_gguf_8tb2e48z`:   0%|          | 0/4 [00:00<?, ?it/s]
Unsloth: Copying 4 files from cache to `/tmp/unsloth_gguf_8tb2e48z`:  25%|██▌       | 1/4 [00:07<00:22,  7.45s/it]
Unsloth: Copying 4 files from cache to `/tmp/unsloth_gguf_8tb2e48z`:  50%|█████     | 2/4 [00:27<00:29, 14.68s/it]
Unsloth: Copying 4 files from cache to `/tmp/unsloth_gguf_8tb2e48z`:  75%|███████▌  | 3/4 [00:47<00:17, 17.04s/it]
Unsloth: Copying 4 files from cache to `/tmp/unsloth_gguf_8tb2e48z`: 100%|██████████| 4/4 [00:59<00:00, 14.84s/it]


Successfully copied all 4 files from cache to `/tmp/unsloth_gguf_8tb2e48z`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.



Unsloth: Preparing safetensor model files: 100%|██████████| 4/4 [00:00<00:00, 104857.60it/s]

Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [01:36<00:00, 24.04s/it]


Unsloth: Merge process complete. Saved to `/tmp/unsloth_gguf_8tb2e48z`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q8_0'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...


RuntimeError: Failed to convert model to GGUF: Unsloth: GGUF conversion failed: Unsloth: Failed to convert vision projector to GGUF with command `/usr/bin/python3 /root/.unsloth/llama.cpp/unsloth_convert_hf_to_gguf.py --outfile Qwen3.5-9B.BF16-mmproj.gguf --outtype bf16 --mmproj --split-max-size 50G /tmp/unsloth_gguf_8tb2e48z`: Command '['/usr/bin/python3', '/root/.unsloth/llama.cpp/unsloth_convert_hf_to_gguf.py', '--outfile', 'Qwen3.5-9B.BF16-mmproj.gguf', '--outtype', 'bf16', '--mmproj', '--split-max-size', '50G', '/tmp/unsloth_gguf_8tb2e48z']' returned non-zero exit status 1.

In [ ]:
import os

# 1. Сохраняем объединенные веса (Base + LoRA) в формате Hugging Face на Google Drive
hf_save_dir = "/content/drive/MyDrive/MITS/checkpoints/mits-tutor-hf-16bit"
print(f"💾 Сохраняем полную VLM модель (HF формат) в {hf_save_dir}...")

# Метод merged_16bit сливает адаптеры с базой и сохраняет всё "как есть", включая Vision-модули
model.save_pretrained_merged(hf_save_dir, tokenizer, save_method="merged_16bit")
print("✅ HF веса успешно сохранены локально!")

# 2. (Опционально) Сразу пушим эту рабочую HF-версию на ваш Hugging Face
HF_REPO_HF = "Siesher/mits-tutor-qwen3.5-9b-hf"
print(f"🚀 Пушим полную модель на Hugging Face ({HF_REPO_HF})...")
try:
    model.push_to_hub_merged(HF_REPO_HF, tokenizer, save_method="merged_16bit")
    print("✅ Модель успешно загружена на Hub!")
except Exception as e:
    print(f"⚠️ Ошибка при пуше на Hub: {e}. Но локальная копия на Drive в безопасности.")

💾 Сохраняем полную VLM модель (HF формат) в /content/drive/MyDrive/MITS/checkpoints/mits-tutor-hf-16bit...


RuntimeError: Unsloth: Failed saving locally - no disk space left. Uploading can work luckily! Use .push_to_hub instead.

In [ ]:
# 🚀 Спасаем модель: отправляем напрямую в облако Hugging Face, минуя ваш диск

HF_REPO_HF = "Siesher/mits-tutor-qwen3.5-9b-hf" # Ваше имя пользователя и название модели
print(f"☁️ Начинаем потоковую загрузку 16-битной VLM модели на Hugging Face: {HF_REPO_HF}...")

try:
    # Метод push_to_hub_merged будет собирать модель кусками и сразу отправлять в интернет,
    # не требуя 20 ГБ свободного места локально.
    model.push_to_hub_merged(
        HF_REPO_HF,
        tokenizer,
        save_method="merged_16bit",
        private=True # Делаем репозиторий приватным (только для вас)
    )
    print("✅ ФУХ! Модель успешно спасена и загружена на Hugging Face!")
    print(f"🔗 Ссылка: https://huggingface.co/{HF_REPO_HF}")

except Exception as e:
    print(f"❌ Ошибка при загрузке: {e}")

☁️ Начинаем потоковую загрузку 16-битной VLM модели на Hugging Face: Siesher/mits-tutor-qwen3.5-9b-hf...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...n3.5-9b-hf/tokenizer.json: 100%|##########| 20.0MB / 20.0MB            

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0004-of-00004.safetensors:   0%|          |  152kB / 3.33GB            


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [04:25<00:00, 66.31s/it]


Unsloth: Merge process complete. Saved to `/content/Siesher/mits-tutor-qwen3.5-9b-hf`
✅ ФУХ! Модель успешно спасена и загружена на Hugging Face!
🔗 Ссылка: https://huggingface.co/Siesher/mits-tutor-qwen3.5-9b-hf
